In [1]:
pip install pandas openpyxl xlrd numpy unidecode


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


## Bloque de procesamiento

Procesamiento/estandarización de una o más fuentes para el dataset final.

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [2]:
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
import pandas as pd
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
import numpy as np
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
from typing import Union
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
from unidecode import unidecode
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
from datetime import datetime, timedelta
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
import re
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
from pandas.tseries.offsets import MonthEnd
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
from functools import reduce
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
from pathlib import Path
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
import os
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
import glob


## Bloque de procesamiento

Procesamiento/estandarización de una o más fuentes para el dataset final.

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [3]:
print (type(pd))


<class 'module'>


#Vamos a trabajar con las bases de datos provenientes del BCU que vienen en formatos particulares y difíciles de trabajar. De cada una de ellas, seleccionaremos algunas variables y armaremos los datasets con el formato adecuado para luego combinar todas las series


## Pasivos/Activos Monetarios (BCU)

Variables monetarias diarias: emisión, depósitos, agregados (M1/M2), base monetaria.

**Archivo(s)/fuente(s):**
- `Bases originales/PAM_diarios.xls`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [4]:
##Comencemos con los Pasivos y activos monetarios. Nos va a
##interesar quedarnos con la emisión, los depósitos a la vista en MN
##El M1, el M1', el M2, la base monetaria y la base monetaria restringida

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_PAM=pd.read_excel("Bases originales/PAM_diarios.xls", sheet_name="saldos diarios", header=[2])
df_PAM.head()
#Eliminamos la primera fila y la primera columna, para que quede bien

df_PAM=df_PAM.iloc[:,1:]

df_PAM=df_PAM.iloc[1:,:]

df_PAM=df_PAM.reset_index(drop=True)

df_PAM.head()


print(df_PAM.columns)
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_PAM.columns = df_PAM.columns.str.strip()
# Crear nuevo DataFrame solo con esas columnas

# Lista de columnas que te interesan (como aparecen en el Excel)
cols_keep = [
    "Fecha",
    "Emisión",
    "Depósitos a la vista en moneda nacional",
    "M1",
    "M1'",
    "M2",
    "Base monetaria",
    "Base monetaria restringida"
]

# Crear nuevo DataFrame solo con esas columnas
df_PAM = df_PAM[cols_keep].copy()
ren = {
    "Emisión":   "emision",
    "Depósitos a la vista en moneda nacional":    "dep_vista_MN"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_PAM = df_PAM.rename(columns={k:v for k,v in ren.items() if k in df_PAM.columns})
df_PAM.head()
#Creamos la variable del mult. monetario, y eliminamos las variables del M1' y la base monetaria restringida
df_PAM["mult_monetario"]=(df_PAM["M1'"]/df_PAM["Base monetaria restringida"])
df_PAM=df_PAM.drop(columns=["M1'","Base monetaria restringida"])
df_PAM.to_excel("Datasets finales/PAM_DB.xlsx", index=False, float_format="%.2f")
df_PAM.head()


Index(['Fecha', 'Emisión', 'Circulante en poder del sistema bancario',
       'Circulante fuera del sistema bancario',
       'Depósitos a la vista en moneda nacional', 'M1',
       'Dep. caja de ahorros en  moneda nacional', 'M1'',
       'Depósitos plazo en moneda nacional', 'M2', 'Base monetaria',
       'Base monetaria restringida', 'Unnamed: 13', 'Unnamed: 14'],
      dtype='object')


,Fecha,emision,dep_vista_MN,M1,M2,Base monetaria,mult_monetario
0,2010-01-01,31980.089940,37552.766226,61863.276642,95063.349321,43426.081302,2.02927
1,2010-01-02,31980.089940,37552.766226,61863.276642,95063.349321,43426.081302,2.02927
2,2010-01-03,31980.089940,37552.766226,61863.276642,95063.349321,43426.081302,2.02927
3,2010-01-04,32569.563245,39579.698295,64015.020511,98940.662597,45439.894034,2.04906
4,2010-01-05,33021.900971,38820.360541,64352.407772,99916.14168,45427.866074,2.008857


## Balances monetarios.

Creamos 2 bases a partir del mismo dataset. Uno para los datos del sistema bancario, y otro para los datos del BCU

**Archivo(s)/fuente(s):**
- `Bases originales/balances_monetarios_consolidados.xls`

Procesamiento/estandarización de una o más fuentes para el dataset final.

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [ ]:
##Sigamos con los balances monetarios. Vamos a tener que crear 2 bases para un mismo dataset original
##Una para el BCU y otra para el sistema bancario. Luego uniremos todo en una única base


PATH  = "Bases originales/balances_monetarios_consolidados.xls"
SHEET = "SIST.BRIO."           # <- tu hoja

# -------- helpers robustos --------
MESES_ES = ['ene','feb','mar','abr','may','jun','jul','ago','sep','oct','nov','dic']
MES_MAP  = {'ene':'jan','feb':'feb','mar':'mar','abr':'apr','may':'may','jun':'jun',
            'jul':'jul','ago':'aug','sep':'sep','oct':'oct','nov':'nov','dic':'dec'}

# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
def excel_serial_to_datetime(x):
    """Convierte número de fecha Excel a pandas.Timestamp (suponiendo base 1899-12-30)."""
    try:
        if pd.isna(x):
            return None
        if isinstance(x, (int, float)) and 20000 <= x <= 60000:  # rango razonable 1954–2064
            return (pd.Timestamp('1899-12-30') + pd.to_timedelta(int(x), unit='D'))
    except Exception:
        pass
    return None

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def looks_like_month(x):
    """True si la celda parece encabezado de mes (ene-10, 2010-01, fecha Excel, Timestamp…)."""
    if isinstance(x, (pd.Timestamp, datetime)):
        return True
    # Reutilización: aplicamos `excel_serial_to_datetime()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    num_dt = excel_serial_to_datetime(x)
    if num_dt is not None:
        return True
    s = str(x).strip().lower().replace('.', '')
    if not s or s in {"nan", "none"}:
        return False
    # ej "ene-10", "dic-2009"
    if any(s.startswith(m) for m in MESES_ES):
        return True
    # intentos genéricos
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    if pd.to_datetime(s, errors="coerce") is not None and not pd.isna(pd.to_datetime(s, errors="coerce")):
        return True
    return False

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def normalize_header_to_month_end(x):
    """Devuelve Timestamp fin de mes desde cualquiera de los formatos de arriba; None si no."""
    if isinstance(x, (pd.Timestamp, datetime)):
        dt = pd.Timestamp(x)
        return dt.to_period("M").to_timestamp("M")
    # Reutilización: aplicamos `excel_serial_to_datetime()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    num_dt = excel_serial_to_datetime(x)
    if num_dt is not None:
        return num_dt.to_period("M").to_timestamp("M")
    s = str(x).strip().lower().replace('.', '')
    # traducir "ene"->"jan" para %b-%y
    for es, en in MES_MAP.items():
        if s.startswith(es):
            s = s.replace(es, en, 1)
            break
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    dt = pd.to_datetime(s, format="%b-%y", errors="coerce")
    if pd.isna(dt):
        # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
        dt = pd.to_datetime(s, errors="coerce")
    if pd.isna(dt):
        return None
    return dt.to_period("M").to_timestamp("M")

# -------- 1) leer sin header --------
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel(PATH, sheet_name=SHEET, header=None, engine="xlrd")

# -------- 2) ubicar fila de meses --------
scores = raw.applymap(looks_like_month).sum(axis=1)
header_row = int(scores.idxmax())

# -------- 3) construir DataFrame con ese header --------
cols_raw = raw.iloc[header_row].tolist()
data = raw.iloc[header_row+1:].copy()                # datos debajo del header
# convertir encabezados a fechas donde corresponda
cols_conv = []
for c in cols_raw:
    # Reutilización: aplicamos `normalize_header_to_month_end()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    m = normalize_header_to_month_end(c)
    cols_conv.append(m if m is not None else (str(c).strip() if not pd.isna(c) else ""))

data.columns = cols_conv

# localizar/renombrar la columna de conceptos (suele ser la 1ª no vacía que NO sea fecha)
concept_idx = 0
for j, c in enumerate(data.columns):
    if not isinstance(c, pd.Timestamp) and (str(c) != ""):
        concept_idx = j
        break

# recortar columnas vacías a la izquierda
if concept_idx > 0:
    data = data.iloc[:, concept_idx:]
# renombrar la 1ª columna a Concepto
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
data = data.rename(columns={data.columns[0]: "Concepto"})

# quitar columnas totalmente vacías y duplicadas
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
data = data.dropna(axis=1, how="all")
data = data.loc[:, ~data.columns.duplicated()].reset_index(drop=True)

# columnas de fechas (Timestamp) = meses
date_cols = [c for c in data.columns if isinstance(c, pd.Timestamp)]

# si no detectó nada, imprimimos para diagnosticar
if not date_cols:
    print("No se detectaron columnas de meses. Primeras columnas:", data.columns[:15].tolist())

# quedarnos solo con Concepto + meses
data = data[["Concepto"] + date_cols]

# -------- 4) tidy: long -> wide --------
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
long = data.melt(id_vars="Concepto", var_name="Fecha", value_name="Valor").dropna(subset=["Fecha", "Valor"])
df_BM_sist = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    long.pivot_table(index="Fecha", columns="Concepto", values="Valor", aggfunc="first")
        .sort_index()
        .reset_index()
)

print("✅ df_BM_sist listo")
print("Rango de fechas:", df_BM_sist["Fecha"].min(), "→", df_BM_sist["Fecha"].max())
print("Columnas (muestra):", df_BM_sist.columns[:8].tolist())
df_BM_sist.head()
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_BM_sist.columns = df_BM_sist.columns.str.strip()
print(df_BM_sist.columns)

cols_keep = [
    "Fecha",
    "A.  Activos externos netos",
    "I. Crédito neto al sector público no financiero",
    "II. Crédito neto al BCU",
    "III. Crédito al sector privado residente",
    "IV.  Letras de regulación monetaria"
]

# Crear nuevo DataFrame solo con esas columnas
df_BM_sist = df_BM_sist[cols_keep].copy()
df_BM_sist.columns.name = None
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_sist = df_BM_sist.rename(columns={"A.  Activos externos netos": "act_ext_sist_brio"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_sist = df_BM_sist.rename(columns={"I. Crédito neto al sector público no financiero": "cred_int_publ_sist_brio"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_sist = df_BM_sist.rename(columns={"II. Crédito neto al BCU": "cred_neto_al_BCU"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_sist = df_BM_sist.rename(columns={"III. Crédito al sector privado residente": "cred_priv_res_sist_brio"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_sist = df_BM_sist.rename(columns={"IV.  Letras de regulación monetaria": "LRM_sist_brio"})
df_BM_sist.head()

##Repetimos el procedimiento para el BCU

SHEET = "BCU"

# -------- 1) leer sin header --------
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel(PATH, sheet_name=SHEET, header=None, engine="xlrd")

# -------- 2) ubicar fila de meses --------
scores = raw.applymap(looks_like_month).sum(axis=1)
header_row = int(scores.idxmax())

# -------- 3) construir DataFrame con ese header --------
cols_raw = raw.iloc[header_row].tolist()
data = raw.iloc[header_row+1:].copy()                # datos debajo del header
# convertir encabezados a fechas donde corresponda
cols_conv = []
for c in cols_raw:
    # Reutilización: aplicamos `normalize_header_to_month_end()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    m = normalize_header_to_month_end(c)
    cols_conv.append(m if m is not None else (str(c).strip() if not pd.isna(c) else ""))

data.columns = cols_conv

# localizar/renombrar la columna de conceptos (suele ser la 1ª no vacía que NO sea fecha)
concept_idx = 0
for j, c in enumerate(data.columns):
    if not isinstance(c, pd.Timestamp) and (str(c) != ""):
        concept_idx = j
        break

# recortar columnas vacías a la izquierda
if concept_idx > 0:
    data = data.iloc[:, concept_idx:]
# renombrar la 1ª columna a Concepto
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
data = data.rename(columns={data.columns[0]: "Concepto"})

# quitar columnas totalmente vacías y duplicadas
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
data = data.dropna(axis=1, how="all")
data = data.loc[:, ~data.columns.duplicated()].reset_index(drop=True)

# columnas de fechas (Timestamp) = meses
date_cols = [c for c in data.columns if isinstance(c, pd.Timestamp)]

# si no detectó nada, imprimimos para diagnosticar
if not date_cols:
    print("No se detectaron columnas de meses. Primeras columnas:", data.columns[:15].tolist())

# quedarnos solo con Concepto + meses
data = data[["Concepto"] + date_cols]

# -------- 4) tidy: long -> wide --------
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
long = data.melt(id_vars="Concepto", var_name="Fecha", value_name="Valor").dropna(subset=["Fecha", "Valor"])
df_BM_BCU = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    long.pivot_table(index="Fecha", columns="Concepto", values="Valor", aggfunc="first")
        .sort_index()
        .reset_index()
)

print("✅ df_BM_BCU listo")
print("Rango de fechas:", df_BM_BCU["Fecha"].min(), "→", df_BM_BCU["Fecha"].max())
print("Columnas (muestra):", df_BM_BCU.columns[:8].tolist())
df_BM_BCU.head()

# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_BM_BCU.columns = df_BM_BCU.columns.str.strip()
print(df_BM_BCU.columns)

cols_keep = [
    "Fecha",
    "A.  Activos externos netos",
    "I. Crédito neto al sector público no financiero",
    "II. Crédito neto al sistema bancario",
    "III. Crédito al sector privado residente",
    "IV.  Letras de regulación monetaria en poder de residentes"
]

# Crear nuevo DataFrame solo con esas columnas
df_BM_BCU = df_BM_BCU[cols_keep].copy()
df_BM_BCU.columns.name = None
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_BCU = df_BM_BCU.rename(columns={"A.  Activos externos netos": "act_ext_BCU"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_BCU = df_BM_BCU.rename(columns={"I. Crédito neto al sector público no financiero": "cred_int_publ_BCU"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_BCU = df_BM_BCU.rename(columns={"II. Crédito neto al sistema bancario": "cred_neto_al_sist_brio"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_BCU = df_BM_BCU.rename(columns={"III. Crédito al sector privado residente": "cred_priv_res_BCU"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_BCU = df_BM_BCU.rename(columns={"IV.  Letras de regulación monetaria en poder de residentes": "LRM_res_BCU"})
df_BM_BCU.head()

##Por último, lo hacemos para el consolidado

SHEET = "CONSOLIDADO "

# -------- 1) leer sin header --------
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel(PATH, sheet_name=SHEET, header=None, engine="xlrd")

# -------- 2) ubicar fila de meses --------
scores = raw.applymap(looks_like_month).sum(axis=1)
header_row = int(scores.idxmax())

# -------- 3) construir DataFrame con ese header --------
cols_raw = raw.iloc[header_row].tolist()
data = raw.iloc[header_row+1:].copy()                # datos debajo del header
# convertir encabezados a fechas donde corresponda
cols_conv = []
for c in cols_raw:
    # Reutilización: aplicamos `normalize_header_to_month_end()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    m = normalize_header_to_month_end(c)
    cols_conv.append(m if m is not None else (str(c).strip() if not pd.isna(c) else ""))

data.columns = cols_conv

# localizar/renombrar la columna de conceptos (suele ser la 1ª no vacía que NO sea fecha)
concept_idx = 0
for j, c in enumerate(data.columns):
    if not isinstance(c, pd.Timestamp) and (str(c) != ""):
        concept_idx = j
        break

# recortar columnas vacías a la izquierda
if concept_idx > 0:
    data = data.iloc[:, concept_idx:]
# renombrar la 1ª columna a Concepto
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
data = data.rename(columns={data.columns[0]: "Concepto"})

# quitar columnas totalmente vacías y duplicadas
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
data = data.dropna(axis=1, how="all")
data = data.loc[:, ~data.columns.duplicated()].reset_index(drop=True)

# columnas de fechas (Timestamp) = meses
date_cols = [c for c in data.columns if isinstance(c, pd.Timestamp)]

# si no detectó nada, imprimimos para diagnosticar
if not date_cols:
    print("No se detectaron columnas de meses. Primeras columnas:", data.columns[:15].tolist())

# quedarnos solo con Concepto + meses
data = data[["Concepto"] + date_cols]

# -------- 4) tidy: long -> wide --------
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
long = data.melt(id_vars="Concepto", var_name="Fecha", value_name="Valor").dropna(subset=["Fecha", "Valor"])
df_BM_consolidado = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    long.pivot_table(index="Fecha", columns="Concepto", values="Valor", aggfunc="first")
        .sort_index()
        .reset_index()
)

print("✅ df_BM_consolidado listo")
print("Rango de fechas:", df_BM_consolidado["Fecha"].min(), "→", df_BM_consolidado["Fecha"].max())
print("Columnas (muestra):", df_BM_consolidado.columns[:8].tolist())
df_BM_consolidado.head()

# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_BM_consolidado.columns = df_BM_consolidado.columns.str.strip()
print(df_BM_consolidado.columns)

cols_keep = [
    "Fecha",
    "II. Depósitos a plazo en moneda nacional",
    "IV. Depósitos en moneda extranjera"
]

# Crear nuevo DataFrame solo con esas columnas
df_BM_consolidado = df_BM_consolidado[cols_keep].copy()
df_BM_consolidado.columns.name = None
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_consolidado = df_BM_consolidado.rename(columns={"II. Depósitos a plazo en moneda nacional": "dep_plazo_MN"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_BM_consolidado = df_BM_consolidado.rename(columns={"IV. Depósitos en moneda extranjera": "dep_ME"})
df_BM_consolidado.head()


##armamos la base final para estas variables

BM_DB = (
    df_BM_sist
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_BM_BCU, on="Fecha", how="inner")
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_BM_consolidado, on="Fecha", how="inner")
)

# 3) Guardamos como CSV
BM_DB.to_excel("Datasets finales/BM_DB.xlsx", index=False, float_format="%.2f")

print("✅ BM_DB creado y guardado en processed/BM_DB.csv")
print("Columnas finales:", BM_DB.columns.tolist())
print("Rango de fechas:", BM_DB['Fecha'].min(), "→", BM_DB['Fecha'].max())


C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\343277820.py:85: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  scores = raw.applymap(looks_like_month).sum(axis=1)


✅ df_BM_sist listo
Rango de fechas: 2009-12-31 00:00:00 → 2025-07-31 00:00:00
Columnas (muestra): ['Fecha', ' 1.  Gobierno Central', ' 2.  Sector privado no bancario residente', ' En millones de U$S', ' de los cuales:  Depósitos en moneda extranjera del sector privado', '1.   Activos externos de corto plazo', '1.  Circulante en poder del sistema bancario', '1.  Empresas públicas y gobiernos departamentales']
Index(['Fecha', '1.  Gobierno Central',
       '2.  Sector privado no bancario residente', 'En millones de U$S',
       'de los cuales:  Depósitos en moneda extranjera del sector privado',
       '1.   Activos externos de corto plazo',
       '1.  Circulante en poder del sistema bancario',
       '1.  Empresas públicas y gobiernos departamentales',
       '1.  Moneda nacional', '2.   Crédito a sector privado no residente',
       '2.  Depósitos vista en moneda nacional', '2.  Moneda extranjera',
       '2.  Resto del sector público', '2.  Sector privado no bancario',
       '2.1  E

C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\343277820.py:180: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  scores = raw.applymap(looks_like_month).sum(axis=1)


✅ df_BM_BCU listo
Rango de fechas: 2009-12-31 00:00:00 → 2025-07-31 00:00:00
Columnas (muestra): ['Fecha', '         1.  Depósitos vista en moneda nacional', '         2.  Depósitos de encaje remunerado en pesos y UI', '         3.  Depósitos overnight  en moneda nacional', '         4.  Depósitos plazo en moneda nacional', '        1.  Depósitos vista en moneda nacional  ', '        2.  Depósitos overnight en moneda nacional  ', '        3.  Depósitos plazo en moneda nacional']
Index(['Fecha', '1.  Depósitos vista en moneda nacional',
       '2.  Depósitos de encaje remunerado en pesos y UI',
       '3.  Depósitos overnight  en moneda nacional',
       '4.  Depósitos plazo en moneda nacional',
       '1.  Depósitos vista en moneda nacional',
       '2.  Depósitos overnight en moneda nacional',
       '3.  Depósitos plazo en moneda nacional', '1.  Gobierno Central',
       'En millones de U$S', '1.  Activos de reserva', '1.  Moneda nacional',
       '2.  Moneda extranjera', '2.  Otros 

C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\343277820.py:276: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  scores = raw.applymap(looks_like_month).sum(axis=1)


✅ df_BM_consolidado listo
Rango de fechas: 2009-12-31 00:00:00 → 2025-07-31 00:00:00
Columnas (muestra): ['Fecha', ' 1.  Gobierno Central', ' 2.  Sector privado no bancario residente', ' En millones de U$S', ' de los cuales:  Depósitos en moneda extranjera del sector privado', '1.   Activos externos de corto plazo', '1.  Circulante en poder del público', '1.  Empresas públicas y gobiernos departamentales']
Index(['Fecha', '1.  Gobierno Central',
       '2.  Sector privado no bancario residente', 'En millones de U$S',
       'de los cuales:  Depósitos en moneda extranjera del sector privado',
       '1.   Activos externos de corto plazo',
       '1.  Circulante en poder del público',
       '1.  Empresas públicas y gobiernos departamentales',
       '1.  Moneda nacional', '2.   Crédito a sector privado no residente',
       '2.  Depósitos vista en moneda nacional', '2.  Moneda extranjera',
       '2.  Resto del sector público', '2.  Sector privado no bancario',
       '2.1  Empresas púb

## PBI a precios constantes

Serie trimestral/anual de PIB real (millones de pesos constantes) para actividad.

**Archivo(s)/fuente(s):**
- `Bases originales/PBI_pesos_constantes.xlsx`

**Columnas referenciadas/seleccionadas en transformaciones:**
Fecha, PRODUCTO INTERNO BRUTO

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [6]:
##Seguimos con el PBI en millones de pesos constantes

pat_trim = re.compile(r"^(I{1,3}|IV)\s*-?\s*(\d{4})\*?$", re.IGNORECASE)

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def is_quarter_cell(x) -> bool:
    return isinstance(x, str) and bool(pat_trim.match(x.strip()))

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def quarter_to_end(lbl: str) -> pd.Timestamp | None:
    m = pat_trim.match(lbl.strip()) if isinstance(lbl, str) else None
    if not m: 
        return None
    q = {"I":1, "II":2, "III":3, "IV":4}[m.group(1).upper()]
    y = int(m.group(2))
    return pd.Period(f"{y}Q{q}").to_timestamp("Q")  # fin de trimestre

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def collapse_text_row(row) -> str:
    parts = [str(v).strip() for v in row if pd.notna(v) and str(v).strip() != ""]
    return re.sub(r"\s+", " ", " ".join(parts)).strip()

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def clean_concept(s: str) -> str:
    s = str(s).strip()
    # quitar códigos al inicio: P.3, P.51, B.1b, etc.
    s = re.sub(r"^\s*(?:[A-Za-z]\.\d+[A-Za-z]?|P\.\d+[A-Za-z]?)\s+", "", s)
    s = s.replace("*", "")
    s = re.sub(r"\s+", " ", s).strip()
    # opcional: quitar el prefijo "(-) " de Importaciones
    s = s.replace("(-) ", "")
    return s

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw=pd.read_excel("Bases originales/PBI_pesos_constantes.xlsx", header=None)


# 2) detectar la FILA que tiene más trimestres
quarter_counts_by_row = raw.applymap(is_quarter_cell).sum(axis=1)
r_qhead = int(quarter_counts_by_row.idxmax())

# 3) columnas donde hay trimestres en esa fila
quarter_mask = raw.iloc[r_qhead].apply(is_quarter_cell)
quarter_cols_idx = np.where(quarter_mask.values)[0].tolist()
if not quarter_cols_idx:
    raise ValueError("No encontré etiquetas de trimestre en ninguna fila.")

# Tomamos el bloque continuo de columnas de trimestres (por si hay alguna celda vacía intermedia)
left_limit  = min(quarter_cols_idx)
right_limit = max(quarter_cols_idx) + 1

date_labels = raw.iloc[r_qhead, left_limit:right_limit].tolist()
# Reutilización: aplicamos `quarter_to_end()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
date_cols   = [quarter_to_end(x) for x in date_labels]

# 4) valores: filas debajo de la fila de trimestres
values_block = raw.iloc[r_qhead+1:, left_limit:right_limit].copy()
values_block.columns = date_cols

# texto de concepto: columnas a la izquierda del bloque de fechas
text_block = raw.iloc[r_qhead+1:, :left_limit].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
concept = text_block.apply(collapse_text_row, axis=1).rename("Concepto")

# 5) armar base y limpiar
# Consolidación: apilamos fuentes con estructura compatible (mismo set de columnas) para una tabla única.
df_pbi = pd.concat([concept, values_block], axis=1)
# quedarnos con columnas que realmente son fechas
fecha_cols = [c for c in df_pbi.columns if isinstance(c, pd.Timestamp)]
df_pbi = df_pbi.loc[:, ["Concepto"] + fecha_cols]
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df_pbi = df_pbi.dropna(axis=1, how="all").dropna(axis=0, how="all").reset_index(drop=True)

# 6) tidy (largo)
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
pbi_tidy = df_pbi.melt(id_vars="Concepto", var_name="Fecha", value_name="Valor")
pbi_tidy["Valor"] = pd.to_numeric(pbi_tidy["Valor"], errors="coerce")
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
pbi_tidy = pbi_tidy.dropna(subset=["Fecha", "Valor"]).reset_index(drop=True)

# 7) limpiar nombres y pasar a ANCHO (una columna por variable)
pbi_tidy["Concepto"] = pbi_tidy["Concepto"].map(clean_concept)

pbi_wide = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    pbi_tidy.pivot_table(index="Fecha", columns="Concepto", values="Valor", aggfunc="first")
            .sort_index()
            .reset_index()
)
pbi_wide.columns.name = None

print("Fila con trimestres detectada:", r_qhead)
print("Rango de fechas:", pbi_tidy["Fecha"].min(), "→", pbi_tidy["Fecha"].max())
print("Columnas ejemplo:", pbi_wide.columns[:10].tolist())
PBI_DB=pbi_wide[["Fecha", "PRODUCTO INTERNO BRUTO"]].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
PBI_DB = PBI_DB.rename(columns={"PRODUCTO INTERNO BRUTO": "PIB"})
PBI_DB.head()
PBI_DB.to_excel("Datasets finales/PBI_DB.xlsx", index=False, float_format="%.2f")


Fila con trimestres detectada: 7
Rango de fechas: 2016-03-31 00:00:00 → 2025-06-30 00:00:00
Columnas ejemplo: ['Fecha', 'Exportaciones de bienes y servicios', 'Formación Bruta de Capital', 'Formación Bruta de Capital Fijo', 'Gasto de Consumo final', 'Gobierno e ISFLSH', 'Hogares', 'Importaciones de bienes y servicios', 'PRODUCTO INTERNO BRUTO', 'Variación de existencias']


C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\771167153.py:39: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  quarter_counts_by_row = raw.applymap(is_quarter_cell).sum(axis=1)


## Sector público: resultado fiscal y recaudación

Ingresos/egresos/inversión y recaudación; variables fiscales.

**Archivo(s)/fuente(s):**
- `Bases originales/Resultados Sector Público Julio 2025.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [7]:
##Sigamos con los resultados del gobierno y la recaudación de impuestos

#Primero, los ingresos, egresos e inversiones del gobierno central

SHEET="Gobierno Central - BPS"
# -------- 1) leer sin header --------
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel("Bases originales/Resultados Sector Público Julio 2025.xlsx", sheet_name=SHEET, header=None)

# -------- 2) ubicar fila de meses --------
scores = raw.applymap(looks_like_month).sum(axis=1)
header_row = int(scores.idxmax())

# -------- 3) construir DataFrame con ese header --------
cols_raw = raw.iloc[header_row].tolist()
data = raw.iloc[header_row+1:].copy()                # datos debajo del header
# convertir encabezados a fechas donde corresponda
cols_conv = []
for c in cols_raw:
    # Reutilización: aplicamos `normalize_header_to_month_end()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    m = normalize_header_to_month_end(c)
    cols_conv.append(m if m is not None else (str(c).strip() if not pd.isna(c) else ""))

data.columns = cols_conv

# localizar/renombrar la columna de conceptos (suele ser la 1ª no vacía que NO sea fecha)
concept_idx = 0
for j, c in enumerate(data.columns):
    if not isinstance(c, pd.Timestamp) and (str(c) != ""):
        concept_idx = j
        break

# recortar columnas vacías a la izquierda
if concept_idx > 0:
    data = data.iloc[:, concept_idx:]
# renombrar la 1ª columna a Concepto
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
data = data.rename(columns={data.columns[0]: "Concepto"})

# quitar columnas totalmente vacías y duplicadas
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
data = data.dropna(axis=1, how="all")
data = data.loc[:, ~data.columns.duplicated()].reset_index(drop=True)

# columnas de fechas (Timestamp) = meses
date_cols = [c for c in data.columns if isinstance(c, pd.Timestamp)]

# si no detectó nada, imprimimos para diagnosticar
if not date_cols:
    print("No se detectaron columnas de meses. Primeras columnas:", data.columns[:15].tolist())

# quedarnos solo con Concepto + meses
data = data[["Concepto"] + date_cols]

# -------- 4) tidy: long -> wide --------
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
long = data.melt(id_vars="Concepto", var_name="Fecha", value_name="Valor").dropna(subset=["Fecha", "Valor"])
df_RES_gob = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    long.pivot_table(index="Fecha", columns="Concepto", values="Valor", aggfunc="first")
        .sort_index()
        .reset_index()
)

print("✅ df_RES_gob listo")
print("Rango de fechas:", df_RES_gob["Fecha"].min(), "→", df_RES_gob["Fecha"].max())
print("Columnas (muestra):", df_RES_gob.columns[:8].tolist())
df_RES_gob.head()
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_RES_gob.columns = df_RES_gob.columns.str.strip()
print(df_RES_gob.columns)

cols_keep = [
    "Fecha",
    "INGRESOS TOTALES GC-BPS",
    "Remuneraciones",
    "Pasividades",
    "Transferencias",
    "Inversiones",
    "Intereses de Deuda Pública"
]

# Crear nuevo DataFrame solo con esas columnas
df_RES_gob = df_RES_gob[cols_keep].copy()
df_RES_gob.columns.name = None
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_RES_gob = df_RES_gob.rename(columns={"INGRESOS TOTALES GC-BPS": "ingresos_gob"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_RES_gob = df_RES_gob.rename(columns={"Remuneraciones": "rem_gob"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_RES_gob = df_RES_gob.rename(columns={"Pasividades": "pas_gob"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_RES_gob = df_RES_gob.rename(columns={"Transferencias": "transf_gob"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_RES_gob = df_RES_gob.rename(columns={"Inversiones": "inv_gob"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_RES_gob = df_RES_gob.rename(columns={"Intereses de Deuda Pública": "int_deuda_gob"})
df_RES_gob.head()

##Ahora hacemos lo mismo para la recaudación de impuestos de la DGI

SHEET="DGI"

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel("Bases originales/Resultados Sector Público Julio 2025.xlsx", sheet_name=SHEET, header=None)

# -------- 2) ubicar fila de meses --------
scores = raw.applymap(looks_like_month).sum(axis=1)
header_row = int(scores.idxmax())

# -------- 3) construir DataFrame con ese header --------
cols_raw = raw.iloc[header_row].tolist()
data = raw.iloc[header_row+1:].copy()                # datos debajo del header
# convertir encabezados a fechas donde corresponda
cols_conv = []
for c in cols_raw:
    # Reutilización: aplicamos `normalize_header_to_month_end()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    m = normalize_header_to_month_end(c)
    cols_conv.append(m if m is not None else (str(c).strip() if not pd.isna(c) else ""))

data.columns = cols_conv

# localizar/renombrar la columna de conceptos (suele ser la 1ª no vacía que NO sea fecha)
concept_idx = 0
for j, c in enumerate(data.columns):
    if not isinstance(c, pd.Timestamp) and (str(c) != ""):
        concept_idx = j
        break

# recortar columnas vacías a la izquierda
if concept_idx > 0:
    data = data.iloc[:, concept_idx:]
# renombrar la 1ª columna a Concepto
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
data = data.rename(columns={data.columns[0]: "Concepto"})

# quitar columnas totalmente vacías y duplicadas
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
data = data.dropna(axis=1, how="all")
data = data.loc[:, ~data.columns.duplicated()].reset_index(drop=True)

# columnas de fechas (Timestamp) = meses
date_cols = [c for c in data.columns if isinstance(c, pd.Timestamp)]

# si no detectó nada, imprimimos para diagnosticar
if not date_cols:
    print("No se detectaron columnas de meses. Primeras columnas:", data.columns[:15].tolist())

# quedarnos solo con Concepto + meses
data = data[["Concepto"] + date_cols]

# -------- 4) tidy: long -> wide --------
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
long = data.melt(id_vars="Concepto", var_name="Fecha", value_name="Valor").dropna(subset=["Fecha", "Valor"])
df_DGI = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    long.pivot_table(index="Fecha", columns="Concepto", values="Valor", aggfunc="first")
        .sort_index()
        .reset_index()
)

print("✅ df_DGI listo")
print("Rango de fechas:", df_DGI["Fecha"].min(), "→", df_DGI["Fecha"].max())
print("Columnas (muestra):", df_DGI.columns[:8].tolist())
df_DGI.head()
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_DGI.columns = df_DGI.columns.str.strip()
print(df_DGI.columns)

cols_keep = [
    "Fecha",
    "I.V.A",
    "IMESI",
    "IRPF",
    "IRIC/IRAE desde ago-07",
    "PATRIMONIO"
]

# Crear nuevo DataFrame solo con esas columnas
df_DGI = df_DGI[cols_keep].copy()
df_DGI.columns.name = None
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_DGI = df_DGI.rename(columns={"I.V.A": "IVA"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_DGI = df_DGI.rename(columns={"IRIC/IRAE desde ago-07": "IRAE"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_DGI = df_DGI.rename(columns={"PATRIMONIO": "imp_patrimonio"})
df_DGI.head()

GOB_DB = (
    df_DGI
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_RES_gob, on="Fecha", how="inner")
)
pd.options.display.float_format = "{:,.2f}".format
GOB_DB=GOB_DB.round(2)

# 3) Guardamos como CSV
GOB_DB.to_excel("Datasets finales/GOB_DB.xlsx", index=False, float_format="%.2f")
GOB_DB.head()

print("✅ GOB_DB creado y guardado en Datasets finales/GOB_DB.xlsx")
print("Columnas finales:", GOB_DB.columns.tolist())
print("Rango de fechas:", GOB_DB['Fecha'].min(), "→", GOB_DB['Fecha'].max())


C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\887115342.py:11: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  scores = raw.applymap(looks_like_month).sum(axis=1)


✅ df_RES_gob listo
Rango de fechas: 1999-01-31 00:00:00 → 2025-07-31 00:00:00
Columnas (muestra): ['Fecha', '- AFAP', '- I.R.P./ IRPF desde ago-07', '- Otros', 'Adm. Central', 'Aportes de Emp. Pcas.', 'Asignaciones Familiares y otras prestaciones activas', 'BPS']
Index(['Fecha', '- AFAP', '- I.R.P./ IRPF desde ago-07', '- Otros',
       'Adm. Central', 'Aportes de Emp. Pcas.',
       'Asignaciones Familiares y otras prestaciones activas', 'BPS',
       'BPS - FSS Ley Nº 19.590 (-)', 'Caja Militar', 'Caja Policial',
       'Cert. Dev. Impuest. BPS (-)', 'Cert. Dev. Impuest. DGI (-)',
       'Comercio Exterior', 'D.G.I.', 'EGRESOS TOTALES GC-BPS',
       'Entes Públicos', 'FIMTOP', 'Gastos No Personales', 'Gobierno Central',
       'INGRESOS TOTALES GC-BPS', 'IRP', 'Ingresos BPS - Recaudación Neta (1)',
       'Ingresos Brutos', 'Ingresos Gobierno Central',
       'Ingresos netos FSS Ley Nº 19.590', 'Intereses de Deuda Pública',
       'Inversiones', 'Loterías', 'MTOP', 'MVOTMA', 'Org. D

C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\887115342.py:108: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  scores = raw.applymap(looks_like_month).sum(axis=1)


✅ df_DGI listo
Rango de fechas: 1999-01-31 00:00:00 → 2025-07-31 00:00:00
Columnas (muestra): ['Fecha', '(-) DOCUMENTOS', 'Anticipo importación (3)', 'Automotores', 'COFIS', 'Categoría I', 'Categoría II (1)', 'Combustibles']
Index(['Fecha', '(-) DOCUMENTOS', 'Anticipo importación (3)', 'Automotores',
       'COFIS', 'Categoría I', 'Categoría II (1)', 'Combustibles', 'I.V.A',
       'IASS', 'ICIR/PRIMARIA I.RURALES*', 'ICOME', 'IMABA y CONTRALOR',
       'IMESI', 'IMESSA', 'IMP. COMISIONES', 'IRA - IMEBA',
       'IRIC/IRAE desde ago-07', 'IRNR', 'IRPF', 'Importación', 'Interno',
       'Interno (1)', 'PATRIMONIO', 'RESTO', 'Resto', 'TOTAL BRUTO',
       'TOTAL NETO (1)', 'TRANS. PATRIMONIALES', 'Tabaco'],
      dtype='object', name='Concepto')
✅ GOB_DB creado y guardado en Datasets finales/GOB_DB.xlsx
Columnas finales: ['Fecha', 'IVA', 'IMESI', 'IRPF', 'IRAE', 'imp_patrimonio', 'ingresos_gob', 'rem_gob', 'pas_gob', 'transf_gob', 'inv_gob', 'int_deuda_gob']
Rango de fechas: 1999-01-31 0

## Sector público: resultado fiscal y recaudación

Ingresos/egresos/inversión y recaudación; variables fiscales.

**Archivo(s)/fuente(s):**
- `Bases originales/Resultados Sector Público Julio 2025.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [8]:
##Ahora para datos provenientes específicamente de ANCAP

SHEET="ANCAP"

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel("Bases originales/Resultados Sector Público Julio 2025.xlsx", sheet_name=SHEET, header=None)

# -------- 2) ubicar fila de meses --------
scores = raw.applymap(looks_like_month).sum(axis=1)
header_row = int(scores.idxmax())

# -------- 3) construir DataFrame con ese header --------
cols_raw = raw.iloc[header_row].tolist()
data = raw.iloc[header_row+1:].copy()                # datos debajo del header
# convertir encabezados a fechas donde corresponda
cols_conv = []
for c in cols_raw:
    # Reutilización: aplicamos `normalize_header_to_month_end()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    m = normalize_header_to_month_end(c)
    cols_conv.append(m if m is not None else (str(c).strip() if not pd.isna(c) else ""))

data.columns = cols_conv

# localizar/renombrar la columna de conceptos (suele ser la 1ª no vacía que NO sea fecha)
concept_idx = 0
for j, c in enumerate(data.columns):
    if not isinstance(c, pd.Timestamp) and (str(c) != ""):
        concept_idx = j
        break

# recortar columnas vacías a la izquierda
if concept_idx > 0:
    data = data.iloc[:, concept_idx:]
# renombrar la 1ª columna a Concepto
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
data = data.rename(columns={data.columns[0]: "Concepto"})

# quitar columnas totalmente vacías y duplicadas
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
data = data.dropna(axis=1, how="all")
data = data.loc[:, ~data.columns.duplicated()].reset_index(drop=True)

# columnas de fechas (Timestamp) = meses
date_cols = [c for c in data.columns if isinstance(c, pd.Timestamp)]

# si no detectó nada, imprimimos para diagnosticar
if not date_cols:
    print("No se detectaron columnas de meses. Primeras columnas:", data.columns[:15].tolist())

# quedarnos solo con Concepto + meses
data = data[["Concepto"] + date_cols]

# -------- 4) tidy: long -> wide --------
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
long = data.melt(id_vars="Concepto", var_name="Fecha", value_name="Valor").dropna(subset=["Fecha", "Valor"])
df_ANCAP = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    long.pivot_table(index="Fecha", columns="Concepto", values="Valor", aggfunc="first")
        .sort_index()
        .reset_index()
)

print("✅ df_ANCAP listo")
print("Rango de fechas:", df_ANCAP["Fecha"].min(), "→", df_ANCAP["Fecha"].max())
print("Columnas (muestra):", df_ANCAP.columns[:8].tolist())
df_ANCAP.head()
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_ANCAP.columns = df_ANCAP.columns.str.strip()
print(df_ANCAP.columns)

cols_keep = [
    "Fecha",
    "Compras de bienes y servicios",
    "Intereses",
    "Variación de stock de petróleo",
    "Inversiones"
]

# Crear nuevo DataFrame solo con esas columnas
df_ANCAP = df_ANCAP[cols_keep].copy()
df_ANCAP.columns.name = None
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_ANCAP = df_ANCAP.rename(columns={"Compras de bienes y servicios": "compras_ANCAP"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_ANCAP = df_ANCAP.rename(columns={"Intereses": "int_ANCAP"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_ANCAP = df_ANCAP.rename(columns={"Variación de stock de petróleo": "var_stock_pet"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_ANCAP = df_ANCAP.rename(columns={"Inversiones": "inv_ANCAP"})
df_ANCAP.head()

ANCAP_DB=df_ANCAP
# 3) Guardamos como CSV
ANCAP_DB.to_excel("Datasets finales/ANCAP_DB.xlsx", index=False, float_format="%.2f")
ANCAP_DB.head()

print("✅ ANCAP_DB creado y guardado en Datasets finales/ANCAP_DB.xlsx")
print("Columnas finales:", ANCAP_DB.columns.tolist())
print("Rango de fechas:", ANCAP_DB['Fecha'].min(), "→", ANCAP_DB['Fecha'].max())


C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\3081886508.py:9: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  scores = raw.applymap(looks_like_month).sum(axis=1)


✅ df_ANCAP listo
Rango de fechas: 2001-01-31 00:00:00 → 2025-07-31 00:00:00
Columnas (muestra): ['Fecha', '       Otras', '       Variación de stock de petróleo', '      Aportes BPS', '      Compras de bienes y servicios  ', '      Impuestos DGI', '      Intereses', '      Inversiones']
Index(['Fecha', 'Otras', 'Variación de stock de petróleo', 'Aportes BPS',
       'Compras de bienes y servicios', 'Impuestos DGI', 'Intereses',
       'Inversiones', 'Remuneraciones', 'Corrientes', 'Dividendo en efectivo',
       'No corrientes', 'Otros Ingresos',
       'Transferencias del Gobierno Central', 'Venta de bienes y servicios',
       'EGRESOS', 'INGRESOS', 'RESULTADO'],
      dtype='object', name='Concepto')
✅ ANCAP_DB creado y guardado en Datasets finales/ANCAP_DB.xlsx
Columnas finales: ['Fecha', 'compras_ANCAP', 'int_ANCAP', 'var_stock_pet', 'inv_ANCAP']
Rango de fechas: 2001-01-31 00:00:00 → 2025-07-31 00:00:00


## Hist_CUTIF_Dolar

Fuente externa: se estandariza para integrarla al dataset macro final.

**Archivo(s)/fuente(s):**
- `Bases originales/Hist_CUTIF_Dolar.xlsx`
- `Bases originales/Hist_CUD.xlsx`
- `Bases originales/Hist_UST.xlsx`
- `Bases originales/Hist_CUI.xlsx`
- `Bases originales/Hist_ITLUP.xlsx`
- `Bases originales/Hist_CEI.xlsx`
- `Bases originales/IRUBEVSA.xlsx`

**Columnas referenciadas/seleccionadas en transformaciones:**
FECHA, VALOR

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [9]:
##Ahora, vamos a descargar, modificar y unir en una única base todas las curvas

##Cutif
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_cutif=pd.read_excel("Bases originales/Hist_CUTIF_Dolar.xlsx", sheet_name=0, header=1)
ren = {
    "FECHA":   "Fecha",
    "1 MES":    "cutif_1mes",
    "2 MESES": "cutif_2meses",
    "3 MESES": "cutif_3meses",
    "4 MESES": "cutif_4meses",
    "5 MESES": "cutif_5meses",
    "6 MESES": "cutif_6meses",
    "7 MESES": "cutif_7meses",
    "8 MESES": "cutif_8meses",
    "9 MESES": "cutif_9meses",
    "10 MESES": "cutif_10meses",
    "11 MESES": "cutif_11meses",
    "12 MESES": "cutif_12meses",
    "13 MESES": "cutif_13meses",
    "14 MESES": "cutif_14meses",
    "15 MESES": "cutif_15meses"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_cutif = df_cutif.rename(columns={k:v for k,v in ren.items() if k in df_cutif.columns})
df_cutif.head()

##CUD

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_cud=pd.read_excel("Bases originales/Hist_CUD.xlsx", sheet_name=0, header=0)
ren = {
    "FECHA":   "Fecha",
    "3 MESES":  "cud_3meses",
    "6 MESES": "cud_6meses",
    "1 AÑO": "cud_1año",
    "2 AÑOS": "cud_2años",
    "3 AÑOS": "cud_3años",
    "4 AÑOS": "cud_4años",
    "5 AÑOS": "cud_5años",
    "6 AÑOS": "cud_6años",
    "7 AÑOS": "cud_7años",
    "8 AÑOS": "cud_8años",
    "9 AÑOS": "cud_9años",
    "10 AÑOS": "cud_10años",
    "15 AÑOS": "cud_15años",
    "20 AÑOS": "cud_20años",
    "25 AÑOS": "cud_25años",
    "30 AÑOS": "cud_30años"
}
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_cud = df_cud.rename(columns={k:v for k,v in ren.items() if k in df_cud.columns})
df_cud["pend_riesgo_sob"] = df_cud["cud_10años"] - df_cud["cud_2años"]
df_cud.head()

##UST

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_ust=pd.read_excel("Bases originales/Hist_UST.xlsx", sheet_name=0, header=0)
ren = {
    "FECHA":   "Fecha",
    "3 MESES":  "ust_3meses",
    "6 MESES": "ust_6meses",
    "1 AÑO": "ust_1año",
    "2 AÑOS": "ust_2años",
    "3 AÑOS": "ust_3años",
    "4 AÑOS": "ust_4años",
    "5 AÑOS": "ust_5años",
    "6 AÑOS": "ust_6años",
    "7 AÑOS": "ust_7años",
    "8 AÑOS": "ust_8años",
    "9 AÑOS": "ust_9años",
    "10 AÑOS": "ust_10años",
    "15 AÑOS": "ust_15años",
    "20 AÑOS": "ust_20años",
    "25 AÑOS": "ust_25años",
    "30 AÑOS": "ust_30años"
}
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_ust = df_ust.rename(columns={k:v for k,v in ren.items() if k in df_ust.columns})
df_ust["proxy_ciclo_global"] = df_ust["ust_10años"] - df_ust["ust_2años"]
df_ust.head()

df_ust_cud=(
    df_ust
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_cud, on="Fecha", how="inner")
)

df_ust_cud["riesgo_sob"]=df_ust_cud["cud_5años"]-df_ust_cud["ust_5años"]
df_ust_cud.head()

##CUI

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_cui=pd.read_excel("Bases originales/Hist_CUI.xlsx", sheet_name=0, header=0)
ren = {
    "FECHA":   "Fecha",
    "3 MESES":  "cui_3meses",
    "6 MESES": "cui_6meses",
    "1 AÑO": "cui_1año",
    "2 AÑOS": "cui_2años",
    "3 AÑOS": "cui_3años",
    "4 AÑOS": "cui_4años",
    "5 AÑOS": "cui_5años",
    "6 AÑOS": "cui_6años",
    "7 AÑOS": "cui_7años",
    "8 AÑOS": "cui_8años",
    "9 AÑOS": "cui_9años",
    "10 AÑOS": "cui_10años",
    "15 AÑOS": "cui_15años",
    "20 AÑOS": "cui_20años",
    "25 AÑOS": "cui_25años",
    "30 AÑOS": "cui_30años"
}
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_cui = df_cui.rename(columns={k:v for k,v in ren.items() if k in df_cui.columns})
df_cui.head()

##ITLUP

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_itlup=pd.read_excel("Bases originales/Hist_ITLUP.xlsx", sheet_name=0, header=0)
ren = {
    "FECHA":   "Fecha",
    "1 MES": "itlup_1mes",
    "2 MESES":  "itlup_2meses",
    "3 MESES":  "itlup_3meses",
    "6 MESES": "itlup_6meses",
    "9 MESES": "itlup_9meses",
    "1 AÑO": "itlup_1año",
    "2 AÑOS": "itlup_2años",
    "3 AÑOS": "itlup_3años",
    "4 AÑOS": "itlup_4años",
    "5 AÑOS": "itlup_5años",
    "6 AÑOS": "itlup_6años",
    "7 AÑOS": "itlup_7años",
    "8 AÑOS": "itlup_8años",
    "9 AÑOS": "itlup_9años",
    "10 AÑOS": "itlup_10años"
}
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_itlup = df_itlup.rename(columns={k:v for k,v in ren.items() if k in df_itlup.columns})
df_itlup.head()

df_cui_itlup=(
    df_cui
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_itlup, on="Fecha", how="inner")
)

df_cui_itlup.head()

##Curva de expectativas inflacionarias

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_cei=pd.read_excel("Bases originales/Hist_CEI.xlsx", sheet_name=0, header=0)
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_cei = df_cei.rename(columns={"FECHA": "Fecha"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_cei = df_cei.rename(columns={"1 AÑO": "cei_1año"})

##IRUBEVSA

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_irubevsa=pd.read_excel("Bases originales/IRUBEVSA.xlsx", sheet_name=0, header=0)
df_irubevsa=df_irubevsa[["FECHA","VALOR"]].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_irubevsa = df_irubevsa.rename(columns={"FECHA": "Fecha"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_irubevsa = df_irubevsa.rename(columns={"VALOR": "IRUBEVSA"})


CURVAS_DB=(df_cui_itlup
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_ust_cud, on="Fecha", how="inner")
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_cutif, on="Fecha", how="inner")
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_cei, on="Fecha", how="inner")
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_irubevsa, on="Fecha", how="inner")
)

# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
CURVAS_DB["Fecha"] = pd.to_datetime(CURVAS_DB["Fecha"], errors="coerce")
CURVAS_DB=CURVAS_DB.drop(columns=["ÍNDICE_x", "PBS", "ÍNDICE_y"])
CURVAS_DB.to_excel("Datasets finales/CURVAS_DB.xlsx", index=False, float_format="%.2f")
CURVAS_DB.head()

print("✅ CURVAS_DB creado y guardado en Datasets finales/CURVAS_DB.xlsx")
print("Columnas finales:", CURVAS_DB.columns.tolist())
print("Rango de fechas:", CURVAS_DB['Fecha'].min(), "→", CURVAS_DB['Fecha'].max())


✅ CURVAS_DB creado y guardado en Datasets finales/CURVAS_DB.xlsx
Columnas finales: ['Fecha', 'cui_3meses', 'cui_6meses', 'cui_1año', 'cui_2años', 'cui_3años', 'cui_4años', 'cui_5años', 'cui_6años', 'cui_7años', 'cui_8años', 'cui_9años', 'cui_10años', 'cui_15años', 'cui_20años', 'cui_25años', 'cui_30años', 'itlup_1mes', 'itlup_2meses', 'itlup_3meses', 'itlup_6meses', 'itlup_9meses', 'itlup_1año', 'itlup_2años', 'itlup_3años', 'itlup_4años', 'itlup_5años', 'itlup_6años', 'itlup_7años', 'itlup_8años', 'itlup_9años', 'itlup_10años', 'ust_3meses', 'ust_6meses', 'ust_1año', 'ust_2años', 'ust_3años', 'ust_4años', 'ust_5años', 'ust_6años', 'ust_7años', 'ust_8años', 'ust_9años', 'ust_10años', 'ust_15años', 'ust_20años', 'ust_25años', 'ust_30años', 'proxy_ciclo_global', 'cud_3meses', 'cud_6meses', 'cud_1año', 'cud_2años', 'cud_3años', 'cud_4años', 'cud_5años', 'cud_6años', 'cud_7años', 'cud_8años', 'cud_9años', 'cud_10años', 'cud_15años', 'cud_20años', 'cud_25años', 'cud_30años', 'pend_riesgo_so

## IMAE (INE/BCU)

Índice mensual de actividad económica.

**Archivo(s)/fuente(s):**
- `Bases originales/IMAE.xlsx`

**Columnas referenciadas/seleccionadas en transformaciones:**
Desestacionalizado, Fecha

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [10]:
##IMAE

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_imae=pd.read_excel("Bases originales/IMAE.xlsx", sheet_name=0, header=0)
df_imae=df_imae[["Fecha", "Desestacionalizado"]]
##Vamos a incluir la var. porcentual mensual (el crecimiento) del índice
df_imae["var_imae"]=100*(np.log(df_imae["Desestacionalizado"]) - np.log(df_imae["Desestacionalizado"].shift(1)))
df_imae=df_imae.drop(columns=["Desestacionalizado"])
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_imae["Fecha"] = pd.to_datetime(df_imae["Fecha"], dayfirst=True, errors="coerce")
df_imae["Fecha"] = df_imae["Fecha"].dt.floor("D")

df_imae.head()
IMAE_DB=df_imae
IMAE_DB.to_excel("Datasets finales/IMAE_DB.xlsx", index=False, float_format="%.2f")



## IPC Uruguay (INE)

Índice de precios al consumo e inflación.

**Archivo(s)/fuente(s):**
- `Bases originales/IPC_Uruguay.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [11]:
##IPC

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_ipc=pd.read_excel("Bases originales/IPC_Uruguay.xlsx", sheet_name=0, header=0)
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df_ipc = df_ipc.dropna(how="all")

# 3) Asegurar 'Año' y 'Mes' numéricos (coerce para eliminar 'Fuente: ...')
df_ipc["Año"] = pd.to_numeric(df_ipc["Año"], errors="coerce")
df_ipc["Mes"] = pd.to_numeric(df_ipc["Mes"], errors="coerce")

# 4) Quedarse solo con filas válidas
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df_ipc = df_ipc.dropna(subset=["Año", "Mes"]).copy()

# 5) Filtrar desde 2015
df_ipc = df_ipc[df_ipc["Año"] >= 2015].copy()

# 6) Mes como entero
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
df_ipc["Mes"] = df_ipc["Mes"].astype(int)

# 7) Construir 'Fecha' = fin de mes (yyyy-mm-dd 00:00:00)
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
dt = pd.to_datetime(dict(year=df_ipc["Año"].astype(int),
                    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
                    month=df_ipc["Mes"].astype(int),
                    day=1), errors="coerce")

df_ipc["Fecha"] = (dt + MonthEnd(0)).dt.floor("D")
df_ipc=df_ipc.drop(columns=["Año", "Mes", "General Montevideo", "General Interior"])
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_ipc=df_ipc.rename(columns={"General Total Pais":"IPC"})
df_ipc = df_ipc[["Fecha"] + [c for c in df_ipc.columns if c != "Fecha"]]
df_ipc.head()
IPC_DB=df_ipc
IPC_DB.to_excel("Datasets finales/IPC_DB.xlsx", index=False, float_format="%.2f")


## Índice de salario real (INE)

Salario real como variable de mercado laboral.

**Archivo(s)/fuente(s):**
- `Bases originales/indice_salario_real.xls`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [12]:
##Índice de salario real

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_ISR=pd.read_excel("Bases originales/indice_salario_real.xls", sheet_name=0, header=[6])

# 4) Quedarse solo con filas válidas
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df_ISR = df_ISR.dropna(subset=["Mes y año"]).copy()
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_ISR["Mes y año"] = pd.to_datetime(df_ISR["Mes y año"], errors="coerce")

# 5) Filtrar desde 2015
df_ISR = df_ISR[df_ISR["Mes y año"].dt.year>= 2015].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_ISR=df_ISR.rename(columns={"Mes y año":"Fecha"})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_ISR["Fecha"] = pd.to_datetime(df_ISR["Fecha"], errors="coerce") + MonthEnd(0)
# (opcional) asegurar 00:00:00 exacto
df_ISR["Fecha"] = df_ISR["Fecha"].dt.floor("D")
df_ISR=df_ISR.drop(columns=["Mensual", "Acum.año", "Acum.12 meses"])
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_ISR=df_ISR.rename(columns={"Índice":"ISR"})
df_ISR.head()
ISR_DB=df_ISR
ISR_DB.to_excel("Datasets finales/ISR_DB.xlsx", index=False, float_format="%.2f")


## Índice de precios del productor

Precios a nivel productor/industria.

**Archivo(s)/fuente(s):**
- `Bases originales/precios_productor.xls`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [13]:
##IPPN


# =============== Helpers ===============
MES_MAP = {
    'ene':'jan','feb':'feb','mar':'mar','abr':'apr','may':'may','jun':'jun',
    'jul':'jul','ago':'aug','sep':'sep','oct':'oct','nov':'nov','dic':'dec'
}

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def mes_ano_to_date(x):
    """Convierte 'ene-88', 'mar-2010', etc. a Timestamp (primer día del mes)."""
    s = str(x).strip().lower().replace('.', '')
    for es, en in MES_MAP.items():
        if s.startswith(es):
            s = s.replace(es, en, 1)
            break
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    dt = pd.to_datetime(s, format="%b-%y", errors="coerce")
    if pd.isna(dt):
        # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
        dt = pd.to_datetime(s, errors="coerce")
    return dt

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def norm_txt(t):
    """Normaliza texto de encabezados (acentos/ñ, espacios, mayúsculas)."""
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return ""
    t = str(t).strip()
    rep = {
        "á":"a","é":"e","í":"i","ó":"o","ú":"u",
        "Á":"A","É":"E","Í":"I","Ó":"O","Ú":"U",
        "ñ":"n","Ñ":"N"
    }
    for a,b in rep.items():
        t = t.replace(a,b)
    return t.replace(".", "").replace("  ", " ").replace(" ", "_").lower()

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def flatten3(col):
    """Aplana 3 niveles de header → 'ippn_plaza__variaciones__mensual'."""
    # Reutilización: aplicamos `norm_txt()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    a, b, c = [norm_txt(x) for x in col]
    parts = [p for p in (a,b,c) if p and not p.startswith("unnamed")]
    return parts[-1] if parts == ["mes_y_ano"] else "__".join(parts)

# =============== Carga robusta ===============
path = "Bases originales/precios_productor.xls"

# 1) Localizar la fila que dice “Mes y año”
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
tmp = pd.read_excel(path, header=None)
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
hdr = tmp.index[tmp.iloc[:,0].astype(str).str.contains("mes y año", case=False, na=False)].tolist()[0]

# 2) Leer con 3 niveles de encabezado (bloque / sub-bloque / variable)
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df0 = pd.read_excel(path, header=[hdr-2, hdr-1, hdr])

# 3) Propagar rótulos hacia la derecha cuando hay celdas combinadas
mi = df0.columns
fixed = []
last = ["","",""]
for a,b,c in mi.to_list():
    aa = a if (pd.notna(a) and not str(a).lower().startswith("unnamed")) else last[0]
    bb = b if (pd.notna(b) and not str(b).lower().startswith("unnamed")) else last[1]
    cc = c
    fixed.append((aa, bb, cc))
    last = [aa, bb, cc]
df0.columns = pd.MultiIndex.from_tuples(fixed)

# 4) Aplanar nombres + limpiar vacíos
# Reutilización: aplicamos `flatten3()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
df0.columns = [flatten3(c) for c in df0.columns]
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df0 = df0.dropna(how="all").dropna(axis=1, how="all")

# 5) Construir Fecha (fin de mes) desde la col de “Mes y año”
fecha_col = next((c for c in df0.columns if c.endswith("mes_y_ano")), None)
if fecha_col is None:
    raise ValueError(f"No se halló la columna 'Mes y año'. Encabezados: {df0.columns.tolist()[:12]}")

dt = (df0[fecha_col] if pd.api.types.is_datetime64_any_dtype(df0[fecha_col])
      # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
      else pd.to_datetime(df0[fecha_col].map(mes_ano_to_date), errors="coerce"))
df0["Fecha"] = dt.dt.to_period("M").dt.to_timestamp("M").dt.floor("D")

# 6) Tipar numéricos (por si hay strings) y filtrar 2015+
for c in df0.columns:
    if c not in ["Fecha", fecha_col]:
        df0[c] = pd.to_numeric(df0[c], errors="coerce")
df0 = df0[df0["Fecha"] >= pd.Timestamp("2015-01-01")].copy()

# 7) Renombres canónicos (solo si existen en tu archivo)
ren = {
    "ippn__numero__indice":                "ippn_indice",
    "ippn__variaciones__mensual":          "ippn_mensual",
    "ippn_plaza__numero__indice":          "ippn_plaza_indice",
    "ippn_plaza__variaciones__mensual":    "ippn_plaza_mensual",
    "ippn_exportacion__numero__indice":    "ippn_exp_indice",
    "ippn_exportacion__variaciones__mensual":"ippn_exp_mensual",
}
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df0 = df0.rename(columns={k:v for k,v in ren.items() if k in df0.columns})

# 8) Selección final (lo que vas a usar)
keep = ["Fecha",
        "ippn_plaza_indice"]
keep = [c for c in keep if c in df0.columns]
df_ippn = df0[keep].copy().reset_index(drop=True)
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_ippn=df_ippn.rename(columns={"ippn_plaza_indice":"IPPN"})
df_ippn.head()
DB_IPPN=df_ippn
DB_IPPN.to_excel("Datasets finales/IPPN_DB.xlsx", index=False, float_format="%.2f")


## IVF industria manufacturera

Índices de volumen físico por ramas relevantes (lácteos, carnes, celulosa, etc.).

**Archivo(s)/fuente(s):**
- `Bases originales/IVF_industria_manufacturera.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [14]:
##IVF industria manufacturera: Vamos a quedarnos solo con los índices para la industria de los productos lácteos, frigoríficos (carnes),
##cueros y la industria de la celulosa y papel, que son las más importantes en el país

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_IVF=pd.read_excel("Bases originales/IVF_industria_manufacturera.xlsx", sheet_name=0, header=[4])
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df_IVF= df_IVF.dropna(how="all")
# 1) Limpiar columnas "Unnamed"
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
df_IVF = df_IVF.loc[:, ~df_IVF.columns.astype(str).str.startswith("Unnamed")].copy()

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def norm_mes(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().lower().replace(".", "")
    for a,b in {"á":"a","é":"e","í":"i","ó":"o","ú":"u"}.items():
        s = s.replace(a,b)
    # unificar ambas grafías
    s = s.replace("septiembre", "setiembre")
    return s

df_IVF["MES"] = df_IVF["MES"].apply(norm_mes)

# mapear a número de mes (soporta completo o abreviado)
map_full = {
    "enero":1,"febrero":2,"marzo":3,"abril":4,"mayo":5,"junio":6,
    "julio":7,"agosto":8,"setiembre":9,"octubre":10,"noviembre":11,"diciembre":12
}
map_abbr = {"ene":1,"feb":2,"mar":3,"abr":4,"may":5,"jun":6,"jul":7,"ago":8,"set":9,"sep":9,"oct":10,"nov":11,"dic":12}

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def mes_to_num(s):
    if pd.isna(s): return np.nan
    return map_full.get(s, map_abbr.get(s[:3], np.nan))

df_IVF["mes_num"] = df_IVF["MES"].apply(mes_to_num)

# aviso si quedó algo sin mapear
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
no_mapeados = df_IVF.loc[df_IVF["mes_num"].isna(), "MES"].dropna().unique()
if len(no_mapeados):
    print("⚠️ Meses no mapeados:", no_mapeados)

# --- AÑO numérico ---
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
df_IVF["AÑO"] = pd.to_numeric(df_IVF["AÑO"], errors="coerce").astype("Int64")

# --- construir Fecha = último día del mes (00:00:00) ---
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df_IVF = df_IVF.dropna(subset=["AÑO","mes_num"]).copy()
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
dt = pd.to_datetime(
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    dict(year=df_IVF["AÑO"].astype(int),
        # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
        month=df_IVF["mes_num"].astype(int),
        day=1),
    errors="coerce"
)
df_IVF["Fecha"] = (dt + MonthEnd(0)).dt.floor("D")

# --- poner Fecha primero ---
df_IVF = df_IVF[["Fecha"] + [c for c in df_IVF.columns if c != "Fecha"]]
keep = ["Fecha",
        101,
        105,
        15,
        17]
df_IVF=df_IVF[keep].copy()
ren = {
    101:   "IVF_carne",
    105:   "IVF_lacteo",
    15:    "IVF_cuero",
    17:    "IVF_pap_y_cel",
}
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_IVF = df_IVF.rename(columns={k:v for k,v in ren.items() if k in df_IVF.columns})
df_IVF.head()

DB_IVF=df_IVF
DB_IVF.to_excel("Datasets finales/IVF_DB.xlsx", index=False, float_format="%.2f")


## UI y UP

"Curva" de unidades indexadas y de unidades previsionales

**Archivo(s)/fuente(s):**
- `Bases originales/UI.xlsx`
- `Bases originales/UP.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [15]:
##UI y UP

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_UI=pd.read_excel("Bases originales/UI.xlsx", sheet_name=0, header=0)
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_UI=df_UI.rename(columns={"FECHA":"Fecha"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_UI=df_UI.rename(columns={"VALOR":"UI"})
df_UI.head()

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_UP=pd.read_excel("Bases originales/UP.xlsx", sheet_name=0, header=0)
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_UP=df_UP.rename(columns={"FECHA":"Fecha"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_UP=df_UP.rename(columns={"VALOR":"UP"})
df_UP.head()

DB_UNIDADES=(df_UP
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_UI, on="Fecha", how="right")
)
DB_UNIDADES.to_excel("Datasets finales/UNIDADES_DB.xlsx", index=False, float_format="%.2f")
DB_UNIDADES.head()


,Fecha,UP,UI
0,2025-09-20,1.71,6.36
1,2025-09-19,1.71,6.36
2,2025-09-18,1.71,6.36
3,2025-09-17,1.71,6.36
4,2025-09-16,1.71,6.36


## TPM

Tasa de política monetaria, definida por el BCU. Es el instrumento de política monetaria adoptado desde el 2020

**Archivo(s)/fuente(s):**
- `Bases originales/TPM.xlsx`

**Columnas referenciadas/seleccionadas en transformaciones:**
Fecha, Tasa de Política Monetaria

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [16]:
##Tasa de política monetaria

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_TPM=pd.read_excel("Bases originales/TPM.xlsx", sheet_name=0, header=0)
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_TPM["Fecha"] = pd.to_datetime(df_TPM["Fecha"], dayfirst=True, errors="coerce")
df_TPM["Fecha"] = df_TPM["Fecha"].dt.floor("D")
df_TPM=df_TPM[["Fecha", "Tasa de Política Monetaria"]].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_TPM=df_TPM.rename(columns={"Tasa de Política Monetaria": "TPM (%)"})
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
tpm = (df_TPM["TPM (%)"].astype(str)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        .str.replace(r'[%\s]', '', regex=True)   # quita % y espacios
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        .str.replace('.', '', regex=False)       # elimina separador de miles (si hubiera)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        .str.replace(',', '.', regex=False))
df_TPM["TPM (%)"] = pd.to_numeric(tpm, errors="coerce")
df_TPM.head()

DB_TPM=df_TPM
DB_TPM.to_excel("Datasets finales/TPM_DB.xlsx", index=False, float_format="%.2f")


## Riesgo país (EMBI)

Spread soberano como proxy de riesgo.

**Archivo(s)/fuente(s):**
- `Bases originales/Serie_Historica_Spread_del_EMBI.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [17]:
##Riesgo país

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_riesgo=pd.read_excel("Bases originales/Serie_Historica_Spread_del_EMBI.xlsx", sheet_name=0, header=1)
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_riesgo["Fecha"] = pd.to_datetime(df_riesgo["Fecha"], dayfirst=True, errors="coerce")
df_riesgo["Fecha"] = df_riesgo["Fecha"].dt.floor("D")

keep = ["Fecha",
        "Argentina",
        "Brasil",
        "Uruguay",
        ]
df_riesgo=df_riesgo[keep].copy()
ren = {
    "Argentina":   "riesgo_AR",
    "Brasil":   "riesgo_BR",
    "Uruguay":    "riesgo_UY",
}
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_riesgo = df_riesgo.rename(columns={k:v for k,v in ren.items() if k in df_riesgo.columns})
df_riesgo.head()

DB_RIESGO=df_riesgo
DB_RIESGO.to_excel("Datasets finales/RIESGO_DB.xlsx", index=False, float_format="%.2f")


## Brasil: series macro (diarias)

TPM/IPC/actividad u otras series de Brasil para shocks externos.

**Archivo(s)/fuente(s):**
- `Bases originales/brasil_daily.csv`
- `Bases originales/brasil_monthly.csv`

**Columnas referenciadas/seleccionadas en transformaciones:**
Fecha, IA_br, IPC_br, TPM_br, inflacion_12m_br, usd_br

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [18]:
##TPM, IPC e índice de actividad para brasil
path = "Bases originales/brasil_daily.csv"

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_TPMbr = pd.read_csv(
    path,
    sep=";",              # separador real
    encoding="latin1",    # común en CSVs de Brasil
    engine="python",
    decimal=",",          # coma decimal → 1,23
    thousands=".",        # separador de miles
    on_bad_lines="skip",  # salta filas rotas
)
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_TPMbr["date"] = pd.to_datetime(df_TPMbr["date"], errors="coerce")
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_TPMbr=df_TPMbr.rename(columns={"date":"Fecha"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_TPMbr=df_TPMbr.rename(columns={"tasa_SELIC":"TPM_br"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_TPMbr=df_TPMbr.rename(columns={"TC_BRL_USD":"usd_br"})
df_TPMbr=df_TPMbr[["Fecha","TPM_br","usd_br"]].copy()
df_TPMbr.head()

path = "Bases originales/brasil_monthly.csv"

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_IPCbr = pd.read_csv(
    path,
    sep=";",              # separador real
    encoding="latin1",    # común en CSVs de Brasil
    engine="python",
    decimal=",",          # coma decimal → 1,23
    thousands=".",        # separador de miles
    on_bad_lines="skip",  # salta filas rotas
)

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_IPCbr=df_IPCbr.rename(columns={"date":"Fecha"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_IPCbr=df_IPCbr.rename(columns={"ipca_indice":"IPC_br"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_IPCbr=df_IPCbr.rename(columns={"inflacion_12m_brasil":"inflacion_12m_br"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_IPCbr=df_IPCbr.rename(columns={"ibcbr_sa":"IA_br"})
df_IPCbr=df_IPCbr[["Fecha","IPC_br", "IA_br", "inflacion_12m_br"]].copy()
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_IPCbr["Fecha"] = pd.to_datetime(df_IPCbr["Fecha"], errors="coerce")
df_IPCbr["Fecha"] = (df_IPCbr["Fecha"] + MonthEnd(0)).dt.floor("D")
df_IPCbr.head()


##EPU de Brasil
PATH = "Bases originales/Brazil_Policy_Uncertainty_Data.xlsx"  # ajusta si tu path es otro

# --- leer la primera hoja ---
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df0 = pd.read_excel(PATH, sheet_name=0)

# normalizar nombres para buscar columnas con flexibilidad
norm = lambda s: re.sub(r"\s+", " ", str(s).strip().lower())

# Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
cols = {c: norm(c) for c in df0.columns}
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df = df0.rename(columns=cols)

# --- detectar columna(s) de fecha ---
fecha = None

# Caso A: Year + Month (o similares)
year_col  = next((c for c in df.columns if re.fullmatch(r"(year|ano|año)", c)), None)
month_col = next((c for c in df.columns if re.fullmatch(r"(month|mes)", c)), None)

if year_col and month_col:
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    fecha = pd.to_datetime(
        # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
        dict(year=df[year_col].astype(int, errors="ignore"),
             # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
             month=df[month_col].astype(int, errors="ignore"),
             day=1),
        errors="coerce"
    )

# Caso B: una única columna de fecha (date, time, period, etc.)
if fecha is None or fecha.notna().mean() < 0.5:
    date_col = next((c for c in df.columns if re.search(r"(date|fecha|time|period)", c)), None)
    if date_col:
        # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
        fecha = pd.to_datetime(df[date_col], errors="coerce")

if fecha is None:
    raise RuntimeError("No pude identificar columna(s) de fecha (Year+Month o Date).")

# llevar la fecha al FIN DE MES
fecha_fem = fecha.dt.to_period("M").dt.to_timestamp("M")

# --- detectar columna EPU ---
# (busca 'epu' o 'uncertainty' en el nombre; ajusta si tu hoja usa otro nombre)
epu_col = next((c for c in df.columns if ("epu" in c or "uncertainty" in c)), None)
if epu_col is None:
    # si no hay nombre claro, intenta elegir la primera numérica “candidata”
    num_candidates = [c for c in df.columns if df[c].dtype.kind in "if"]
    if not num_candidates:
        raise RuntimeError("No encontré columna EPU (ni columnas numéricas candidatas).")
    epu_col = num_candidates[0]

# asegurar numérico
epu = pd.to_numeric(df[epu_col], errors="coerce")

# --- construir el dataframe final ---
df_EPU = (
    pd.DataFrame({"Fecha": fecha_fem, "EPU": epu})
      # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
      .dropna(subset=["Fecha", "EPU"])
      .sort_values("Fecha")
      .reset_index(drop=True)
)

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_EPU=df_EPU.rename(columns={"EPU":"EPU_br"})


DB_BRASIL=(df_TPMbr
        # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
        .merge(df_IPCbr, on="Fecha", how="left")
        # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
        .merge(df_EPU, on="Fecha", how="left")
)
DB_BRASIL=DB_BRASIL[DB_BRASIL["Fecha"] >= pd.Timestamp("2015-01-01")].copy()
DB_BRASIL.to_excel("Datasets finales/BRASIL_DB.xlsx", index=False, float_format="%.2f")


## FRED (St. Louis Fed)

Series macro internacionales descargadas de FRED.

**Archivo(s)/fuente(s):**
- `Bases originales/fred_series.csv`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [19]:
##Datos bajados desde Fred

path = "Bases originales/fred_series.csv"

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_fred = pd.read_csv(
    path,
    sep=";",              # separador real
    encoding="latin1",    # común en CSVs de Brasil
    engine="python",
    decimal=",",          # coma decimal → 1,23
    thousands=".",        # separador de miles
    on_bad_lines="skip",  # salta filas rotas
)

df_fred.head()
##Para elegir con que variables nos quedamos, seguimos las recomendaciones de chatgpt para evitar redundancia y correlación muy alta entre predictores
keep = ["Date",
        "VIX",
        "WTI",
        "CPI",
        "CorePCE",
        "FedFunds",
        "NFCI",
        "UnemploymentRate",
        "M2_USA",
        "USD_NominalBroad",
        "SOFR",
        "Breakeven10Y",
        "STLFSI2"
        ]
df_fred=df_fred[keep].copy()
ren = {
    "Date":   "Fecha",
    "CPI":     "IPC_usa",
    "CorePCE": "core_pce_USA",
    "UnemploymentRate": "tasa_desempleo_usa",
    "Breakeven10Y": "prima_implicita_inflacion_10años"
}
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_fred = df_fred.rename(columns={k:v for k,v in ren.items() if k in df_fred.columns})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_fred["Fecha"] = pd.to_datetime(df_fred["Fecha"], errors="coerce")
df_fred.head()

DB_FRED=df_fred
DB_FRED.to_excel("Datasets finales/FRED_DB.xlsx", index=False, float_format="%.2f")



## Yahoo Finance

Series financieras/commodities descargadas de Yahoo Finance.

**Archivo(s)/fuente(s):**
- `Bases originales/yahoo_series.csv`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [20]:
##Datos de Yahoo Finance

path = "Bases originales/yahoo_series.csv"

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_yahoo = pd.read_csv(
    path,
    sep=";",              # separador real
    encoding="utf-8-sig", 
    engine="python",
    decimal=",",          # coma decimal → 1,23
    thousands=".",        # separador de miles
    on_bad_lines="skip"
)

df_yahoo.head(10)

keep = ["Date",
        "UUP_proxy",
        "MOVE",
        "BOVESPA",
        "EEM_EM",
        "ILF_LatAm",
        "Gold",
        "Soy",
        "Rice",
        "LiveCattle",
        "RJA",
        "SP500",
        "MERVAL_Arg",
        "USD_CLP",
        "USD_PEN",
        "USD_MXN",
        "USD_COP",
        "USD_PYG",
        "USD_INR",
        "USD_IDR",
        "USD_CNY",
        "USD_KRW",
        "USD_THB",
        "USD_JPY",
        "USD_EUR",
        "USD_GBP",
        "USD_CHF",
        "USD_CAD",
        "USD_AUD",
        "USD_NZD",
        "USD_ZAR",
        "DXY_ICE"
        ]

df_yahoo=df_yahoo[keep].copy()
ren = {
    "Date":   "Fecha",
    "ILF_LatAm":    "ILF_lat",
    "Gold": "precio_oro",
    "Soy" : "precio_soja",
    "Rice": "precio_arroz",
    "LiveCattle": "precio_carne",
    "RJA": "indice_precios_commodities",
    "MERVAL_Arg": "MERVAL_index",
    "USD_CLP": "dolar_chile",
    "USD_PEN": "dolar_peru",
    "USD_MXN": "dolar_mexico",
    "USD_COP": "dolar_colombia",
    "USD_PYG": "dolar_paraguay",
    "USD_INR": "dolar_india",
    "USD_IDR": "dolar_indonesia",
    "USD_CNY": "dolar_china",
    "USD_KRW": "dolar_corea",
    "USD_THB":"dolar_tailandia",
    "USD_JPY": "dolar_japon",
    "USD_EUR": "dolar_euro",
    "USD_GBP": "dolar_libra",
    "USD_CHF": "dolar_suiza",
    "USD_CAD": "dolar_canada",
    "USD_AUD": "dolar_australia",
    "USD_NZD": "dolar_nueva_zelanda",
    "USD_ZAR": "dolar_sudafrica",
    "DXY_ICE":"DXY_index"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_yahoo = df_yahoo.rename(columns={k:v for k,v in ren.items() if k in df_yahoo.columns})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_yahoo["Fecha"] = pd.to_datetime(df_yahoo["Fecha"], errors="coerce")

DB_YAHOO=df_yahoo
DB_YAHOO.to_excel("Datasets finales/YAHOO_DB.xlsx", index=False, float_format="%.2f")


## Reservas internacionales (BCU)

Stock de reservas internacionales.

**Archivo(s)/fuente(s):**
- `Bases originales/Reservas_BCU.xls`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [21]:
##Reservas del BCU

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_reservas=pd.read_excel("Bases originales/Reservas_BCU.xls", sheet_name=0, header=5)
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
df_reservas= df_reservas.loc[:, ~df_reservas.columns.astype(str).str.startswith("Unnamed")].copy()
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_reservas.columns = df_reservas.columns.str.replace(r"\s+", " ", regex=True)  # múltiples espacios/saltos -> uno
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_reservas.columns = df_reservas.columns.str.replace("\n", " ", regex=True)
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_reservas.columns = df_reservas.columns.str.strip()
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_reservas["Fecha"] = pd.to_datetime(df_reservas["Fecha"], errors="coerce")
df_reservas = df_reservas.drop(index=0).reset_index(drop=True)
keep = ["Fecha",
        "ACTIVOS DE RESERVA",
        "OBLIGACIONES EN MONEDA EXTRANJERA CON EL SECTOR PUBLICO",
        "OBLIGACIONES EN MONEDA EXTRANJERA CON EL SECTOR FINANCIERO",
        ":POSICION EN MONEDA EXTRANJERA DEL B.C.U."
        ]
df_reservas=df_reservas[keep].copy()
ren = {
    "ACTIVOS DE RESERVA":   "act_res_BCU",
    "OBLIGACIONES EN MONEDA EXTRANJERA CON EL SECTOR PUBLICO":    "obl_ME_SP_BCU",
    "OBLIGACIONES EN MONEDA EXTRANJERA CON EL SECTOR FINANCIERO": "obl_ME_SF_BCU",
    ":POSICION EN MONEDA EXTRANJERA DEL B.C.U." : "pos_ME_BCU"
}
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_reservas = df_reservas.rename(columns={k:v for k,v in ren.items() if k in df_reservas.columns})
df_reservas.head()

DB_RESERVAS=df_reservas
DB_RESERVAS=DB_RESERVAS[DB_RESERVAS["Fecha"] >= pd.Timestamp("2015-01-01")].copy()
DB_RESERVAS.to_excel("Datasets finales/RESERVAS_DB.xlsx", index=False, float_format="%.2f")


## Tipo de Cambio Real Efectivo (BCU)

Índice TCR (base 2019=100) para competitividad.

**Archivo(s)/fuente(s):**
- `Bases originales/TCRE_BCU.xls`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [22]:
##Tipo de cambio real(indicador del BCU, índice base 2019=100)

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_TCR=pd.read_excel("Bases originales/TCRE_BCU.xls", sheet_name=0, header=9)
df_TCR.head()
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
df_TCR= df_TCR.loc[:, ~df_TCR.columns.astype(str).str.startswith("Unnamed: 0")].copy()
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_TCR.columns = df_TCR.columns.str.replace(r"\s+", " ", regex=True)  # múltiples espacios/saltos -> uno
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_TCR.columns = df_TCR.columns.str.replace("\n", " ", regex=True)
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_TCR.columns = df_TCR.columns.str.strip()

keep = ["Unnamed: 1",
        "Global",
        "Extraregional",
        "Regional"
        ]
df_TCR=df_TCR[keep].copy()
ren = {
    "Unnamed: 1":   "Fecha",
    "Global":    "TCR_global",
    "Extraregional": "TCR_extraregional",
    "Regional" : "TCR_regional"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_TCR = df_TCR.rename(columns={k:v for k,v in ren.items() if k in df_TCR.columns})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_TCR ["Fecha"] = pd.to_datetime(df_TCR["Fecha"], errors="coerce")+ MonthEnd(0)
df_TCR.head()

DB_TCR=df_TCR
DB_TCR=DB_TCR[DB_TCR["Fecha"] >= pd.Timestamp("2015-01-01")].copy()
DB_TCR.to_excel("Datasets finales/TCR_DB.xlsx", index=False, float_format="%.2f")



## Encuesta de expectativas de Inflación

Nos quedamos únicamente con el dato de la mediana de la encuesta de expectativas de inflación

**Columnas referenciadas/seleccionadas en transformaciones:**
Fecha, mediana

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [23]:
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def cargar_exp_mes_encuesta(path, solo_metricas=None):
    """
    Lee la encuesta de expectativas y devuelve solo la CATEGORÍA 1
    ('Pronóstico de inflación mensual para el mes de la encuesta').

    solo_metricas: iterable con subcolumnas a conservar (en español, tal como aparecen:
                   'Promedio simple', 'Mediana', 'Desvío estándar', 'Mínimo', 'Máximo').
                   Si es None -> conserva todas.
    """
    # Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
    import pandas as pd, numpy as np, re
    # Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
    from pandas.tseries.offsets import MonthEnd

    # ---------- helpers ----------
    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def norm(s):
        if s is None or (isinstance(s, float) and pd.isna(s)): return ""
        s = str(s).replace("\n"," ").strip().lower()
        s = (s.replace("á","a").replace("é","e").replace("í","i")
               .replace("ó","o").replace("ú","u").replace("ñ","n"))
        s = re.sub(r"\s+"," ",s)
        return s

    MES_MAP = {'ene':'jan','feb':'feb','mar':'mar','abr':'apr','may':'may','jun':'jun',
               'jul':'jul','ago':'aug','sep':'sep','set':'sep','oct':'oct','nov':'nov','dic':'dec'}

    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def parse_mes_ano(x):
        if pd.isna(x): return pd.NaT
        if isinstance(x, pd.Timestamp):
            return x.to_period("M").to_timestamp("M")
        s = str(x).strip().lower().replace(".","")
        for es,en in MES_MAP.items():
            if s.startswith(es): s = s.replace(es,en,1); break
        # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
        dt = pd.to_datetime(s, format="%b-%y", errors="coerce")
        # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
        if pd.isna(dt): dt = pd.to_datetime(s, errors="coerce")
        return dt.to_period("M").to_timestamp("M") if not pd.isna(dt) else pd.NaT

    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def normalize_number_series(s: pd.Series) -> pd.Series:
        # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
        s = s.astype(str).str.replace("\u00A0","", regex=False)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        s = s.str.replace(r"[^0-9,.\-]", "", regex=True)
        # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
        def _fix_one(x):
            if "," in x and "." in x:
                return x.replace(".", "").replace(",", ".")
            if "," in x and "." not in x:
                if x.count(",") > 1:
                    left, right = x.rsplit(",", 1)
                    return left.replace(",", "") + "." + right
                else:
                    return x.replace(",", ".")
            return x
        s = s.apply(_fix_one)
        return pd.to_numeric(s, errors="coerce")

    wanted_map = {
        "promedio simple": "promedio_simple",
        "mediana": "mediana",
        "desvio estandar": "desvio_estandar",
        "desvió estandar": "desvio_estandar",   # por si viene con tilde rara
        "desvío estandar": "desvio_estandar",
        "minimo": "minimo",
        "maximo": "maximo",
    }

    # ---------- leer crudo y detectar filas de header ----------
    # Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
    raw = pd.read_excel(path, header=None, dtype=object)

    # fila del top donde aparece 'Fecha encuesta'
    top_row_idx = next(i for i in range(len(raw))
                       # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
                       if any(("fecha" in norm(v) and "encuesta" in norm(v))
                              for v in raw.iloc[i].tolist()))
    # fila de subtítulos (donde aparece 'Promedio simple')
    sub_row_idx = next(j for j in range(top_row_idx+1, min(top_row_idx+8, len(raw)))
                       # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
                       if any("promedio simple" in norm(v) for v in raw.iloc[j].tolist()))

    # ---------- construir MultiIndex propagando top ----------
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    top_labels = raw.iloc[top_row_idx].astype(str)
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    sub_labels = raw.iloc[sub_row_idx].astype(str)

    fixed_top, last = [], ""
    for v in top_labels:
        # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        vv = "" if (v is None or str(v).lower().startswith("unnamed") or norm(v)=="") else str(v)
        fixed_top.append(last if vv=="" else vv)
        if vv != "": last = vv

    cols = pd.MultiIndex.from_arrays([fixed_top, sub_labels])
    data = raw.iloc[sub_row_idx+1:].copy()
    data.columns = cols

    # ---------- localizar Fecha encuesta ----------
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    top_n = pd.Index([norm(x) for x in data.columns.get_level_values(0)])
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    sub_n = pd.Index([norm(x) for x in data.columns.get_level_values(1)])
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    mask_fecha = (top_n.str.contains("fecha") & top_n.str.contains("encuesta")) | \
                 (sub_n.str.contains("fecha") & sub_n.str.contains("encuesta"))
    fecha_col = list(data.columns[mask_fecha])[0]

    # ---------- CATEGORÍA 1: título exacto del bloque ----------
    titulo_cat1 = "1. pronostico de inflacion mensual para el mes de la encuesta"
    mask_cat1 = (top_n == titulo_cat1) | \
                (top_n.str.contains("pronostico") & top_n.str.contains("mensual") & top_n.str.contains("mes de la encuesta"))

    # subcolumnas deseadas dentro de esa categoría
    mask_subwanted = sub_n.isin(list(wanted_map.keys()))
    cat1_cols = list(data.columns[mask_cat1 & mask_subwanted])

    # Fallback posicional: si en tu archivo el top está cortado, toma las 5 a la derecha de Fecha
    if len(cat1_cols) < 3:
        start = list(data.columns).index(fecha_col)
        take = []
        for k in range(start+1, len(data.columns)):
            t, s = data.columns[k]
            if not str(s).lower().startswith("unnamed"):
                take.append(data.columns[k])
            if len(take) == 5: break
        cat1_cols = take

    # ---------- construir df con Fecha + categoría 1 ----------
    keep_cols = [fecha_col] + cat1_cols
    df = data.loc[:, keep_cols].copy()

    # renombrar a un solo nivel
    new_cols = ["Fecha"]
    for t, s in df.columns[1:]:
        # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        key = norm(s)
        if key not in wanted_map:
            for k in wanted_map:
                if key.startswith(k): key = k; break
        new_cols.append(wanted_map.get(key, key))
    df.columns = new_cols

    # quitar duplicados (por celdas combinadas)
    dups = df.columns.duplicated(keep="first")
    if dups.any(): df = df.loc[:, ~dups].copy()

    # fecha + números
    df["Fecha"] = df["Fecha"].apply(parse_mes_ano)
    for c in [c for c in df.columns if c != "Fecha"]:
        # Reutilización: aplicamos `normalize_number_series()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        df[c] = normalize_number_series(df[c])

    # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
    df = (df.dropna(subset=["Fecha"])
            # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
            .dropna(how="all", subset=[c for c in df.columns if c!="Fecha"])
            .sort_values("Fecha")
            .reset_index(drop=True))

    # filtrar métricas si se pide
    if solo_metricas is not None:
        # normalizamos igual que arriba para mapear nombres
        solo_norm = []
        for m in solo_metricas:
            # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
            k = norm(m)
            if k not in wanted_map:
                for kk in wanted_map:
                    if k.startswith(kk): k = kk; break
            solo_norm.append(wanted_map.get(k, k))
        keep = ["Fecha"] + [c for c in df.columns if c in solo_norm]
        df = df[keep].copy()

    return df

# ======= USO =======
# 1) Todas las métricas de la categoría 1:
# Reutilización: aplicamos `cargar_exp_mes_encuesta()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
df_exp = cargar_exp_mes_encuesta("Bases originales/encuesta_expectativas_inflacion.XLS")
df_exp=df_exp[["Fecha", "mediana"]].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_exp = df_exp.rename(columns={"mediana": "exp_inflacion"})
df_exp.head()
DB_EXP=df_exp
DB_EXP=DB_EXP[DB_EXP["Fecha"] >= pd.Timestamp("2010-01-01")].copy()
DB_EXP.to_excel("Datasets finales/EXP_DB.xlsx", index=False, float_format="%.2f")


## Exportaciones

Exportaciones por país/producto (valores) para sector externo.

**Archivo(s)/fuente(s):**
- `Bases originales/Exportaciones_por producto.xls`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [24]:
##Exportaciones

##Primero, las exp. de 2019 a 2025
# ========= 1) leer con 2 niveles de encabezado =========
path  = "Bases originales/Exportaciones_por producto.xls"
sheet = "Expor_CIIU Rev.3 desde ene2019"   # la hoja que mostraste

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df0 = pd.read_excel(path, sheet_name=sheet, header=[7, 8])

# ========= 2) identificar columnas de fecha/serie =========
lvl0 = pd.Series(df0.columns.get_level_values(0))  # arriba (fechas o 'Unnamed: x')
lvl1 = pd.Series(df0.columns.get_level_values(1))  # abajo (debe decir 'miles de dólares')

# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
is_musd = lvl1.astype(str).str.strip().str.lower().eq("miles de dólares")
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
is_date = pd.to_datetime(lvl0.astype(str), errors="coerce").notna()
series_cols = df0.columns[is_musd & is_date]           # solo series mensuales

# ========= 3) detectar la columna de producto de forma robusta =========
# 3a) si existe 'unnamed: 3' en el nivel superior, usarla
cand = [c for c in df0.columns if re.search(r"unnamed:\s*3", str(c[0]).lower())]
if cand:
    prod_col = cand[0]
else:
    # 3b) fallback: elegir la columna con más "texto no numérico"
    # (entre las que no son series y no parecen fecha)
    non_series = [c for c in df0.columns if c not in series_cols]
    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def text_score(col):
        # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
        s = df0[col].astype(str)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        s = s[~s.str.fullmatch(r"\s*|nan|none", case=False, na=False)]
        # puntuamos por "no numérico" (letras presentes)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        return (s.str.contains(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]", regex=True)).sum()
    # Reutilización: aplicamos `text_score()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    scores = {col: text_score(col) for col in non_series}
    prod_col = max(scores, key=scores.get)

# ========= 4) quedarnos con Producto + series =========
df = df0[[prod_col] + list(series_cols)].copy()

# renombrar a un solo nivel
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df = df.rename(columns={prod_col: "Producto"})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
fechas = pd.to_datetime([c[0] for c in series_cols], errors="coerce")
df.columns = ["Producto"] + list(fechas)

# ========= 5) limpiar filas que no son datos =========
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
df["Producto"] = df["Producto"].astype(str).str.strip()

# quitar filas vacías en todas las fechas
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df = df.dropna(how="all", subset=df.columns[1:])

# quitar cabeceras/separadores típicos
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
bad_mask = df["Producto"].str.lower().str.fullmatch(
    r"(seccion|clase de actividad|grupo de productos)", na=False
)
df = df[~bad_mask].copy()

# ========= 6) largo -> ancho (tidy & pivot) =========
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
long = df.melt(id_vars="Producto", var_name="Fecha", value_name="valor_musd")

# limpiar posibles separadores si vinieran como texto
long["valor_musd"] = (
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    long["valor_musd"].astype(str)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        .str.replace(r"[^\d,.\-]", "", regex=True)
        .apply(lambda x: x.replace(".", "").replace(",", ".") if ("," in x and "." in x)
            else x.replace(",", "."))
)
long["valor_musd"] = pd.to_numeric(long["valor_musd"], errors="coerce")

# descartar filas sin producto o sin valor
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
long = long.dropna(subset=["Producto", "valor_musd"])

# pivot final (si existiera duplicado exacto Fecha-Producto, nos quedamos con el primero no nulo)
df_exports1 = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    long.pivot_table(index="Fecha", columns="Producto", values="valor_musd", aggfunc="first")
        .sort_index()
        .reset_index()
)

# ordenar columnas (Fecha primero)
cols = ["Fecha"] + sorted([c for c in df_exports1.columns if c != "Fecha"])
df_exports1 = df_exports1[cols]

# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_exports1.columns = df_exports1 .columns.str.replace(r"\s+", " ", regex=True)  # múltiples espacios/saltos -> uno
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_exports1.columns = df_exports1 .columns.str.replace("\n", " ", regex=True)
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_exports1.columns = df_exports1 .columns.str.strip()

keep = ["Fecha",
        "Agricultura, Ganadería, Caza y actividades de servicios conexas",
        "Producción, procesamiento y conservación de carne y productos cárnicos",
        "Elaboración de arroz y productos derivados del arroz",
        "Producción de madera y fabricación de productos de madera y corcho excepto muebles; fabricación de artículos de paja y de materiales trenzables",
        "Fabricación de vehículos automotores, remolques y semirremolques; Fabricación de otros tipos de equipo de transporte"
        ]
df_exports1=df_exports1[keep].copy()

ren = {
    "Agricultura, Ganadería, Caza y actividades de servicios conexas":   "exp_agr_y_gan",
    "Producción, procesamiento y conservación de carne y productos cárnicos":    "exp_carne",
    "Elaboración de arroz y productos derivados del arroz": "exp_arroz",
    "Producción de madera y fabricación de productos de madera y corcho excepto muebles; fabricación de artículos de paja y de materiales trenzables" : "exp_madera",
    "Fabricación de vehículos automotores, remolques y semirremolques; Fabricación de otros tipos de equipo de transporte": "exp_veh_y_trans"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_exports1 = df_exports1.rename(columns={k:v for k,v in ren.items() if k in df_exports1.columns})
df_exports1.columns.name = None
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_exports1["Fecha"] = pd.to_datetime(df_exports1["Fecha"], errors="coerce")+ MonthEnd(0)
df_exports1.head()

##Ahora, las exp desde 2018 hacia atrás

# ========= 1) leer con 2 niveles de encabezado =========
path  = "Bases originales/Exportaciones_por producto.xls"
sheet = "Expor_CIIU Rev.3 hasta dic2018"   # la hoja que mostraste

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df0 = pd.read_excel(path, sheet_name=sheet, header=[7, 8])

# ========= 2) identificar columnas de fecha/serie =========
lvl0 = pd.Series(df0.columns.get_level_values(0))  # arriba (fechas o 'Unnamed: x')
lvl1 = pd.Series(df0.columns.get_level_values(1))  # abajo (debe decir 'miles de dólares')

# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
is_musd = lvl1.astype(str).str.strip().str.lower().eq("miles de dólares")
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
is_date = pd.to_datetime(lvl0.astype(str), errors="coerce").notna()
series_cols = df0.columns[is_musd & is_date]           # solo series mensuales

# ========= 3) detectar la columna de producto de forma robusta =========
# 3a) si existe 'unnamed: 3' en el nivel superior, usarla
cand = [c for c in df0.columns if re.search(r"unnamed:\s*3", str(c[0]).lower())]
if cand:
    prod_col = cand[0]
else:
    # 3b) fallback: elegir la columna con más "texto no numérico"
    # (entre las que no son series y no parecen fecha)
    non_series = [c for c in df0.columns if c not in series_cols]
    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def text_score(col):
        # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
        s = df0[col].astype(str)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        s = s[~s.str.fullmatch(r"\s*|nan|none", case=False, na=False)]
        # puntuamos por "no numérico" (letras presentes)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        return (s.str.contains(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]", regex=True)).sum()
    # Reutilización: aplicamos `text_score()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    scores = {col: text_score(col) for col in non_series}
    prod_col = max(scores, key=scores.get)

# ========= 4) quedarnos con Producto + series =========
df = df0[[prod_col] + list(series_cols)].copy()

# renombrar a un solo nivel
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df = df.rename(columns={prod_col: "Producto"})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
fechas = pd.to_datetime([c[0] for c in series_cols], errors="coerce")
df.columns = ["Producto"] + list(fechas)

# ========= 5) limpiar filas que no son datos =========
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
df["Producto"] = df["Producto"].astype(str).str.strip()

# quitar filas vacías en todas las fechas
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df = df.dropna(how="all", subset=df.columns[1:])

# quitar cabeceras/separadores típicos
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
bad_mask = df["Producto"].str.lower().str.fullmatch(
    r"(seccion|clase de actividad|grupo de productos)", na=False
)
df = df[~bad_mask].copy()

# ========= 6) largo -> ancho (tidy & pivot) =========
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
long = df.melt(id_vars="Producto", var_name="Fecha", value_name="valor_musd")

# limpiar posibles separadores si vinieran como texto
long["valor_musd"] = (
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    long["valor_musd"].astype(str)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        .str.replace(r"[^\d,.\-]", "", regex=True)
        .apply(lambda x: x.replace(".", "").replace(",", ".") if ("," in x and "." in x)
            else x.replace(",", "."))
)
long["valor_musd"] = pd.to_numeric(long["valor_musd"], errors="coerce")

# descartar filas sin producto o sin valor
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
long = long.dropna(subset=["Producto", "valor_musd"])

# pivot final (si existiera duplicado exacto Fecha-Producto, nos quedamos con el primero no nulo)
df_exports2 = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    long.pivot_table(index="Fecha", columns="Producto", values="valor_musd", aggfunc="first")
        .sort_index()
        .reset_index()
)

# ordenar columnas (Fecha primero)
cols = ["Fecha"] + sorted([c for c in df_exports2.columns if c != "Fecha"])
df_exports2 = df_exports2[cols]

# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_exports2.columns = df_exports2.columns.str.replace(r"\s+", " ", regex=True)  # múltiples espacios/saltos -> uno
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_exports2.columns = df_exports2.columns.str.replace("\n", " ", regex=True)
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
df_exports2.columns = df_exports2.columns.str.strip()

keep = ["Fecha",
        "Agricultura, Ganadería, Caza y actividades de servicios conexas",
        "Producción, procesamiento y conservación de carne y productos cárnicos",
        "Elaboración de arroz y productos derivados del arroz",
        "Producción de madera y fabricación de productos de madera y corcho excepto muebles; fabricación de artículos de paja y de materiales trenzables",
        "Fabricación de vehículos automotores, remolques y semirremolques; Fabricación de otros tipos de equipo de transporte",
        "Fabricación de sustancias y productos químicos"
        ]
df_exports2=df_exports2[keep].copy()

ren = {
    "Agricultura, Ganadería, Caza y actividades de servicios conexas":   "exp_agr_y_gan",
    "Producción, procesamiento y conservación de carne y productos cárnicos":    "exp_carne",
    "Elaboración de arroz y productos derivados del arroz": "exp_arroz",
    "Producción de madera y fabricación de productos de madera y corcho excepto muebles; fabricación de artículos de paja y de materiales trenzables" : "exp_madera",
    "Fabricación de vehículos automotores, remolques y semirremolques; Fabricación de otros tipos de equipo de transporte": "exp_veh_y_trans",
    "Fabricación de sustancias y productos químicos": "exp_quimicos"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_exports2 = df_exports2.rename(columns={k:v for k,v in ren.items() if k in df_exports2.columns})
df_exports2.columns.name = None
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_exports2["Fecha"] = pd.to_datetime(df_exports2["Fecha"], errors="coerce")+ MonthEnd(0)
df_exports2=df_exports2[df_exports2["Fecha"] >= pd.Timestamp("2010-01-31")].copy()
df_exports2.tail()
##Ahora, simplemente combinamos ambos dataframes en un único dataframe con todas las exportaciones

# Consolidación: apilamos fuentes con estructura compatible (mismo set de columnas) para una tabla única.
DB_EXPORTS = pd.concat([df_exports1, df_exports2], ignore_index=True)
DB_EXPORTS = DB_EXPORTS.sort_values("Fecha").reset_index(drop=True)
DB_EXPORTS.to_excel("Datasets finales/EXPORTS_DB.xlsx", index=False, float_format="%.2f")
DB_EXPORTS.head()


C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\2187771465.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  is_date = pd.to_datetime(lvl0.astype(str), errors="coerce").notna()
C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\2187771465.py:146: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  is_date = pd.to_datetime(lvl0.astype(str), errors="coerce").notna()


,Fecha,exp_agr_y_gan,exp_carne,exp_arroz,exp_madera,exp_veh_y_trans,exp_quimicos
0,2010-01-31,"39,244.00","111,742.00","26,467.00","8,815.00","8,941.00","23,584.00"
1,2010-02-28,"42,488.00","100,665.00","31,597.00","16,657.00","10,670.00","25,938.00"
2,2010-03-31,"54,825.00","117,996.00","17,155.00","15,071.00","14,420.00","33,388.00"
3,2010-04-30,"106,259.00","154,488.00","31,016.00","21,654.00","12,453.00","32,007.00"
4,2010-05-31,"180,293.00","143,559.00","39,020.00","18,322.00","17,257.00","38,618.00"


## Importaciones

Importaciones por origen/producto para sector externo.

**Archivo(s)/fuente(s):**
- `Bases originales/Importaciones_pais_val.xls`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [25]:
##Importaciones por origen

##Primero, empezamos por las importaciones desde el 2019

# ==== 1) Leer primera hoja con header en fila 7 ====
path  = "Bases originales/Importaciones_pais_val.xls"
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df0   = pd.read_excel(path, sheet_name="Impor Origen desde enero 2019", header=7)

# ==== 2) Detectar columnas de fecha (solo esas son series) ====
cols = pd.Index(df0.columns)
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
date_mask = pd.to_datetime(cols.astype(str), errors="coerce").notna()
date_cols = cols[date_mask].tolist()

# ==== 3) Detectar la columna de País de forma robusta ====
# Intento por nombre evidente
cand = [c for c in cols[~date_mask]
        if re.search(r"(pa[ií]s|country|pa[ií]ses|unnamed:\s*3)", str(c), flags=re.I)]
if cand:
    country_col = cand[0]
else:
    # Fallback: entre las no-fecha, elegir la que tenga más texto no numérico
    non_date_cols = cols[~date_mask].tolist()
    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def text_score(series):
        # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
        s = series.astype(str)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        s = s[~s.str.fullmatch(r"\s*|nan|none", case=False, na=False)]
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        return (s.str.contains(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]", regex=True)).sum()
    # Reutilización: aplicamos `text_score()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    scores = {c: text_score(df0[c]) for c in non_date_cols}
    country_col = max(scores, key=scores.get)

# ==== 4) Subset: País + columnas fecha ====
df = df0[[country_col] + date_cols].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df = df.rename(columns={country_col: "Pais"})

# ==== 5) Limpiar filas no-dato y vacías ====
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
df["Pais"] = df["Pais"].astype(str).str.strip()

# quitar filas con todas las fechas NaN
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df = df.dropna(axis=0, how="all", subset=date_cols)

# quitar encabezados/separadores típicos y totales
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
bad_mask = df["Pais"].str.lower().str.contains(
    r"^secci[oó]n$|^grupo de pa[ií]ses$|^zona geogr[aá]fica$|^clase de actividad$|^total|^totales",
    regex=True, na=False
)
df = df[~bad_mask].copy()

# ==== 6) Largo -> Ancho ====
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
long = df.melt(id_vars="Pais", value_vars=date_cols,
               var_name="Fecha", value_name="valor_musd")

# Fechas a datetime (si querés fin de mes, descomenta 2 líneas abajo)
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
long["Fecha"] = pd.to_datetime(long["Fecha"], errors="coerce")
# from pandas.tseries.offsets import MonthEnd
# long["Fecha"] = long["Fecha"].dt.to_period("M").dt.to_timestamp("M")

# Normalizar números por si hubiera separadores de miles/decimales
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def normalize_number_series(s: pd.Series) -> pd.Series:
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    s = s.astype(str).str.replace("\u00A0","", regex=False)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s = s.str.replace(r"[^0-9,.\-]", "", regex=True)
    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def _fix_one(x: str) -> str:
        if "," in x and "." in x:        # '.' miles, ',' decimal
            return x.replace(".", "").replace(",", ".")
        if "," in x and "." not in x:    # solo comas
            if x.count(",") > 1:
                left, right = x.rsplit(",", 1)
                return left.replace(",", "") + "." + right
            return x.replace(",", ".")
        return x
    return pd.to_numeric(s.apply(_fix_one), errors="coerce")

# Reutilización: aplicamos `normalize_number_series()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
long["valor_musd"] = normalize_number_series(long["valor_musd"])

# quitar filas sin país o sin valor
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
long = long.dropna(subset=["Pais", "Fecha", "valor_musd"])

# ==== 7) Pivot final: 1 fila por Fecha, columnas por País ====
df_imports1 = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    long.pivot_table(index="Fecha", columns="Pais", values="valor_musd", aggfunc="first")
        .sort_index()
        .reset_index()
)

# ordenar columnas (Fecha primero)
df_imports1 = df_imports1[["Fecha"] + sorted([c for c in df_imports1.columns if c != "Fecha"])]

keep = ["Fecha",
        "Argentina",
        "Brasil",
        "ESTADOS UNIDOS",
        "U.E. (UNION EUROPEA)",
        "China (Continental)"
        ]
df_imports1=df_imports1[keep].copy()

ren = {
    "Argentina":   "imp_arg",
    "Brasil":    "imp_brl",
    "ESTADOS UNIDOS": "imp_eeuu",
    "U.E. (UNION EUROPEA)" : "imp_EU",
    "China (Continental)": "imp_chn"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_imports1 = df_imports1.rename(columns={k:v for k,v in ren.items() if k in df_imports1.columns})
df_imports1.columns.name = None
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_imports1["Fecha"] = pd.to_datetime(df_imports1["Fecha"], errors="coerce")+ MonthEnd(0)
df_imports1.head()

##Repetimos para las importaciones hasta 2018

##Primero, empezamos por las importaciones desde el 2019

# ==== 1) Leer primera hoja con header en fila 7 ====
path  = "Bases originales/Importaciones_pais_val.xls"
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df0   = pd.read_excel(path, sheet_name="Impor Origen hasta dic 2018", header=7)

# ==== 2) Detectar columnas de fecha (solo esas son series) ====
cols = pd.Index(df0.columns)
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
date_mask = pd.to_datetime(cols.astype(str), errors="coerce").notna()
date_cols = cols[date_mask].tolist()

# ==== 3) Detectar la columna de País de forma robusta ====
# Intento por nombre evidente
cand = [c for c in cols[~date_mask]
        if re.search(r"(pa[ií]s|country|pa[ií]ses|unnamed:\s*3)", str(c), flags=re.I)]
if cand:
    country_col = cand[0]
else:
    # Fallback: entre las no-fecha, elegir la que tenga más texto no numérico
    non_date_cols = cols[~date_mask].tolist()
    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def text_score(series):
        # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
        s = series.astype(str)
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        s = s[~s.str.fullmatch(r"\s*|nan|none", case=False, na=False)]
        # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
        return (s.str.contains(r"[A-Za-zÁÉÍÓÚáéíóúñÑ]", regex=True)).sum()
    # Reutilización: aplicamos `text_score()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    scores = {c: text_score(df0[c]) for c in non_date_cols}
    country_col = max(scores, key=scores.get)

# ==== 4) Subset: País + columnas fecha ====
df = df0[[country_col] + date_cols].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df = df.rename(columns={country_col: "Pais"})

# ==== 5) Limpiar filas no-dato y vacías ====
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
df["Pais"] = df["Pais"].astype(str).str.strip()

# quitar filas con todas las fechas NaN
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
df = df.dropna(axis=0, how="all", subset=date_cols)

# quitar encabezados/separadores típicos y totales
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
bad_mask = df["Pais"].str.lower().str.contains(
    r"^secci[oó]n$|^grupo de pa[ií]ses$|^zona geogr[aá]fica$|^clase de actividad$|^total|^totales",
    regex=True, na=False
)
df = df[~bad_mask].copy()

# ==== 6) Largo -> Ancho ====
# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
long = df.melt(id_vars="Pais", value_vars=date_cols,
               var_name="Fecha", value_name="valor_musd")

# Fechas a datetime (si querés fin de mes, descomenta 2 líneas abajo)
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
long["Fecha"] = pd.to_datetime(long["Fecha"], errors="coerce")
# from pandas.tseries.offsets import MonthEnd
# long["Fecha"] = long["Fecha"].dt.to_period("M").dt.to_timestamp("M")

# Normalizar números por si hubiera separadores de miles/decimales
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def normalize_number_series(s: pd.Series) -> pd.Series:
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    s = s.astype(str).str.replace("\u00A0","", regex=False)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s = s.str.replace(r"[^0-9,.\-]", "", regex=True)
    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def _fix_one(x: str) -> str:
        if "," in x and "." in x:        # '.' miles, ',' decimal
            return x.replace(".", "").replace(",", ".")
        if "," in x and "." not in x:    # solo comas
            if x.count(",") > 1:
                left, right = x.rsplit(",", 1)
                return left.replace(",", "") + "." + right
            return x.replace(",", ".")
        return x
    return pd.to_numeric(s.apply(_fix_one), errors="coerce")

# Reutilización: aplicamos `normalize_number_series()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
long["valor_musd"] = normalize_number_series(long["valor_musd"])

# quitar filas sin país o sin valor
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
long = long.dropna(subset=["Pais", "Fecha", "valor_musd"])

# ==== 7) Pivot final: 1 fila por Fecha, columnas por País ====
df_imports2 = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    long.pivot_table(index="Fecha", columns="Pais", values="valor_musd", aggfunc="first")
        .sort_index()
        .reset_index()
)

# ordenar columnas (Fecha primero)
df_imports2 = df_imports2[["Fecha"] + sorted([c for c in df_imports2.columns if c != "Fecha"])]

keep = ["Fecha",
        "Argentina",
        "Brasil",
        "ESTADOS UNIDOS",
        "U.E. (UNION EUROPEA)",
        "China (Continental)"
        ]
df_imports2=df_imports2[keep].copy()

ren = {
    "Argentina":   "imp_arg",
    "Brasil":    "imp_brl",
    "ESTADOS UNIDOS": "imp_eeuu",
    "U.E. (UNION EUROPEA)" : "imp_EU",
    "China (Continental)": "imp_chn"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_imports2 = df_imports2.rename(columns={k:v for k,v in ren.items() if k in df_imports2.columns})
df_imports2.columns.name = None
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_imports2["Fecha"] = pd.to_datetime(df_imports2["Fecha"], errors="coerce")+ MonthEnd(0)
df_imports2=df_imports2[df_imports2["Fecha"] >= pd.Timestamp("2010-01-31")].copy()
df_imports2.head()

##Combinamos ambos dataframes en un único dataframe con todas las importaciones
# Consolidación: apilamos fuentes con estructura compatible (mismo set de columnas) para una tabla única.
DB_IMPORTS = pd.concat([df_imports1, df_imports2], ignore_index=True)
DB_IMPORTS = DB_IMPORTS.sort_values("Fecha").reset_index(drop=True)
DB_IMPORTS.to_excel("Datasets finales/IMPORTS_DB.xlsx", index=False, float_format="%.2f")


C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\4121186328.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_mask = pd.to_datetime(cols.astype(str), errors="coerce").notna()
C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\4121186328.py:142: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  date_mask = pd.to_datetime(cols.astype(str), errors="coerce").notna()


## Tasas efectivas del sistema bancario

Obtenemos las tasas vigentes en pesos, dólares y UI del sistema financiero para empresas y familias

**Archivo(s)/fuente(s):**
- `Bases originales/tasas_sistema_bancario.xls`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [26]:
##Tasas efectivas del sistema bancario

##Tasas en pesos
PATH  = "Bases originales/tasas_sistema_bancario.xls"
SHEET = "Activas $"

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def norm(s):
    if s is None or (isinstance(s,float) and pd.isna(s)): return ""
    s = str(s).strip().lower().replace("\n"," ")
    s = s.translate(str.maketrans("áéíóúÁÉÍÓÚñÑ","aeiouAEIOUnN"))
    return re.sub(r"\s+"," ",s)

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def excel_serial_to_ts(x):
    if pd.isna(x): return pd.NaT
    if isinstance(x,(int,float)) and 10000 <= x <= 60000:
        return pd.Timestamp("1899-12-30") + pd.to_timedelta(int(x), unit="D")
    return pd.NaT

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def to_float(s: pd.Series) -> pd.Series:
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    s = s.astype(str).str.replace("\u00A0","", regex=False)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s = s.str.replace(r"[^\d,.\-]", "", regex=True)
    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def fix(x):
        if "," in x and "." in x:  # '.' miles, ',' decimal
            return x.replace(".","").replace(",",".")
        if "," in x:
            if x.count(",")>1:
                L,R = x.rsplit(",",1);  return L.replace(",","")+"."+R
            return x.replace(",",".")
        return x
    return pd.to_numeric(s.apply(fix), errors="coerce")

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def parse_fecha(col: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(col): 
        return col
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    dt = pd.to_datetime(col, errors="coerce", dayfirst=True)
    miss = dt.isna()
    if miss.any():
        dt.loc[miss] = col[miss].apply(excel_serial_to_ts)
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    return pd.to_datetime(dt, errors="coerce")

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def flatten_two_level(mi) -> list[str]:
    out=[]
    for a,b in mi.to_list():
        # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        aa,bb = norm(a), norm(b)
        parts=[p for p in (aa,bb) if p and not p.startswith("unnamed")]
        out.append("__".join(parts) if parts else (aa or bb))
    return out

# ---- cargar crudo y detectar fila 'Mes' + encabezado doble ----
xls = pd.ExcelFile(PATH)
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel(xls, sheet_name=SHEET, header=None, dtype=object)

mes_rows = []
for i in range(min(len(raw), 160)):
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    row_norm = [norm(v) for v in raw.iloc[i].tolist()]
    if any(c == "mes" or c.startswith("fecha") for c in row_norm):
        mes_rows.append(i)

if not mes_rows:
    raise ValueError("No encontré ninguna fila con 'Mes' o 'Fecha'.")

chosen = None
two_level = False
for r in mes_rows:
    if r-1 >= 0:
        # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        prev = [norm(v) for v in raw.iloc[r-1].tolist()]
        if any("empres" in c for c in prev) or any("famil" in c for c in prev):
            chosen = (r-1, r); two_level=True; break
if chosen is None:
    chosen = (mes_rows[0],)

# ---- leer y aplanar ----
if two_level:
    # Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
    df0 = pd.read_excel(xls, sheet_name=SHEET, header=list(chosen))
    # Reutilización: aplicamos `flatten_two_level()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    flat = flatten_two_level(df0.columns)
    df0.columns = flat
else:
    # Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
    df0 = pd.read_excel(xls, sheet_name=SHEET, header=chosen[0])

# ---- elegir columnas clave (con heurística por contenido) ----
# Fecha = 'mes' o 'fecha'
# Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
fecha_candidates = [c for c in df0.columns if norm(c) == "mes" or norm(c).startswith("fecha")]
if not fecha_candidates:
    raise RuntimeError("No hallé columna de fecha ('Mes' o 'Fecha').")
fecha_col = fecha_candidates[0]

# Heurística: las filas 0–1 contienen rótulos ("promedio"/"empresas"/"familias"/"plazo ...")
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
r0 = df0.iloc[0].astype(str).str.lower()
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
r1 = df0.iloc[1].astype(str).str.lower()

# Columna de PROMEDIO EMPRESAS: fila0 contiene "promedio" y fila1 contiene "empres"
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
mask_emp = r0.str.contains("promedio", na=False) & r1.str.contains("empres", na=False)
emp_guess = list(df0.columns[mask_emp])

# Columna de PROMEDIO FAMILIAS: fila0 contiene "promedio" y fila1 contiene "famil"
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
mask_fam = r0.str.contains("promedio", na=False) & r1.str.contains("famil", na=False)
fam_guess = list(df0.columns[mask_fam])

# Si la heurística no detectó alguna, usa tu pick_group (con ranking)
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def pick_group(df, group, prefer_general=False):
    g = group.lower()
    names = list(df.columns)
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    n = lambda c: norm(c)

    cols = [c for c in names if n(c).startswith(g)]
    if not cols:
        return None

    exact = [c for c in cols if n(c) == g]
    if exact:
        return exact[0]

    prom = [c for c in cols if "prom" in n(c)]
    if prom:
        return sorted(prom, key=lambda c: len(n(c)))[0]

    bad_words = ("plazo","credito","crédito","creditos","créditos","sobregiro","sobregiros",
                "prestamo","prestamos","por")
    good = [c for c in cols if not any(w in n(c) for w in bad_words)]
    if good:
        return sorted(good, key=lambda c: len(n(c)))[0]

    if prefer_general:
        gen = next((c for c in names if n(c) == "promedio"), None)
        if gen:
            return gen
    return None

# Elegimos columnas finales
# Reutilización: aplicamos `pick_group()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
emp_col = (emp_guess[0] if emp_guess else pick_group(df0, "empresas", prefer_general=False))
# Reutilización: aplicamos `pick_group()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
fam_col = (fam_guess[0] if fam_guess else pick_group(df0, "familias", prefer_general=True))

if emp_col is None or fam_col is None:
    print("[diag] columnas aplanadas:")
    for c in df0.columns: print(" -", c)
    raise RuntimeError("No pude identificar la columna 'promedio' de empresas y/o familias.")

print(f"[sel] fecha_col='{fecha_col}', emp_col='{emp_col}', fam_col='{fam_col}'")

# ---- borrar las dos filas de rótulos si efectivamente aparecieron ----
if (mask_emp.any() or mask_fam.any()) and (
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    r0.str.contains("promedio|plazo", na=False).any() or r1.str.contains("empres|famil|dias|días", na=False).any()
):
    df0 = df0.iloc[2:].reset_index(drop=True)

# ---- construir limpio ----
df_pesos = df0[[fecha_col, emp_col, fam_col]].copy()
df_pesos.columns = ["Fecha", "empresas", "familias"]


df_pesos = (
    # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
    df_pesos.dropna(subset=["Fecha"])
            # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
            .dropna(how="all", subset=["empresas","familias"])
            .sort_values("Fecha")
            .reset_index(drop=True)
)

# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_pesos["Fecha"] = pd.to_datetime(df_pesos["Fecha"], errors="coerce")+ MonthEnd(0)

ren = {
    "empresas":   "tasa_pesos_empresas",
    "familias":    "tasa_pesos_familias"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_pesos = df_pesos.rename(columns={k:v for k,v in ren.items() if k in df_pesos.columns})
df_pesos.columns.name = None

df_pesos.head()

##Tasas en UI
PATH  = "Bases originales/tasas_sistema_bancario.xls"
SHEET = "Activas UI"

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def norm(s):
    if s is None or (isinstance(s,float) and pd.isna(s)): return ""
    s = str(s).strip().lower().replace("\n"," ")
    s = s.translate(str.maketrans("áéíóúÁÉÍÓÚñÑ","aeiouAEIOUnN"))
    return re.sub(r"\s+"," ",s)

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def excel_serial_to_ts(x):
    if pd.isna(x): return pd.NaT
    if isinstance(x,(int,float)) and 10000 <= x <= 60000:
        return pd.Timestamp("1899-12-30") + pd.to_timedelta(int(x), unit="D")
    return pd.NaT

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def to_float(s: pd.Series) -> pd.Series:
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    s = s.astype(str).str.replace("\u00A0","", regex=False)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s = s.str.replace(r"[^\d,.\-]", "", regex=True)
    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def fix(x):
        if "," in x and "." in x:  # '.' miles, ',' decimal
            return x.replace(".","").replace(",",".")
        if "," in x:
            if x.count(",")>1:
                L,R = x.rsplit(",",1);  return L.replace(",","")+"."+R
            return x.replace(",",".")
        return x
    return pd.to_numeric(s.apply(fix), errors="coerce")

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def parse_fecha(col: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(col): 
        return col
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    dt = pd.to_datetime(col, errors="coerce", dayfirst=True)
    miss = dt.isna()
    if miss.any():
        dt.loc[miss] = col[miss].apply(excel_serial_to_ts)
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    return pd.to_datetime(dt, errors="coerce")

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def flatten_two_level(mi) -> list[str]:
    out=[]
    for a,b in mi.to_list():
        # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        aa,bb = norm(a), norm(b)
        parts=[p for p in (aa,bb) if p and not p.startswith("unnamed")]
        out.append("__".join(parts) if parts else (aa or bb))
    return out

# ---- cargar crudo y detectar fila 'Mes' + encabezado doble ----
xls = pd.ExcelFile(PATH)
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel(xls, sheet_name=SHEET, header=None, dtype=object)

mes_rows = []
for i in range(min(len(raw), 160)):
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    row_norm = [norm(v) for v in raw.iloc[i].tolist()]
    if any(c == "mes" or c.startswith("fecha") for c in row_norm):
        mes_rows.append(i)

if not mes_rows:
    raise ValueError("No encontré ninguna fila con 'Mes' o 'Fecha'.")

chosen = None
two_level = False
for r in mes_rows:
    if r-1 >= 0:
        # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        prev = [norm(v) for v in raw.iloc[r-1].tolist()]
        if any("empres" in c for c in prev) or any("famil" in c for c in prev):
            chosen = (r-1, r); two_level=True; break
if chosen is None:
    chosen = (mes_rows[0],)

# ---- leer y aplanar ----
if two_level:
    # Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
    df0 = pd.read_excel(xls, sheet_name=SHEET, header=list(chosen))
    # Reutilización: aplicamos `flatten_two_level()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    flat = flatten_two_level(df0.columns)
    df0.columns = flat
else:
    # Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
    df0 = pd.read_excel(xls, sheet_name=SHEET, header=chosen[0])

# ---- elegir columnas clave (con heurística por contenido) ----
# Fecha = 'mes' o 'fecha'
# Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
fecha_candidates = [c for c in df0.columns if norm(c) == "mes" or norm(c).startswith("fecha")]
if not fecha_candidates:
    raise RuntimeError("No hallé columna de fecha ('Mes' o 'Fecha').")
fecha_col = fecha_candidates[0]

# Heurística: las filas 0–1 contienen rótulos ("promedio"/"empresas"/"familias"/"plazo ...")
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
r0 = df0.iloc[0].astype(str).str.lower()
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
r1 = df0.iloc[1].astype(str).str.lower()

# Columna de PROMEDIO EMPRESAS: fila0 contiene "promedio" y fila1 contiene "empres"
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
mask_emp = r0.str.contains("promedio", na=False) & r1.str.contains("empres", na=False)
emp_guess = list(df0.columns[mask_emp])

# Columna de PROMEDIO FAMILIAS: fila0 contiene "promedio" y fila1 contiene "famil"
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
mask_fam = r0.str.contains("promedio", na=False) & r1.str.contains("famil", na=False)
fam_guess = list(df0.columns[mask_fam])

# Si la heurística no detectó alguna, usa tu pick_group (con ranking)
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def pick_group(df, group, prefer_general=False):
    g = group.lower()
    names = list(df.columns)
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    n = lambda c: norm(c)

    cols = [c for c in names if n(c).startswith(g)]
    if not cols:
        return None

    exact = [c for c in cols if n(c) == g]
    if exact:
        return exact[0]

    prom = [c for c in cols if "prom" in n(c)]
    if prom:
        return sorted(prom, key=lambda c: len(n(c)))[0]

    bad_words = ("plazo","credito","crédito","creditos","créditos","sobregiro","sobregiros",
                "prestamo","prestamos","por")
    good = [c for c in cols if not any(w in n(c) for w in bad_words)]
    if good:
        return sorted(good, key=lambda c: len(n(c)))[0]

    if prefer_general:
        gen = next((c for c in names if n(c) == "promedio"), None)
        if gen:
            return gen
    return None

# Elegimos columnas finales
# Reutilización: aplicamos `pick_group()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
emp_col = (emp_guess[0] if emp_guess else pick_group(df0, "empresas", prefer_general=False))
# Reutilización: aplicamos `pick_group()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
fam_col = (fam_guess[0] if fam_guess else pick_group(df0, "familias", prefer_general=True))

if emp_col is None or fam_col is None:
    print("[diag] columnas aplanadas:")
    for c in df0.columns: print(" -", c)
    raise RuntimeError("No pude identificar la columna 'promedio' de empresas y/o familias.")

print(f"[sel] fecha_col='{fecha_col}', emp_col='{emp_col}', fam_col='{fam_col}'")

# ---- borrar las dos filas de rótulos si efectivamente aparecieron ----
if (mask_emp.any() or mask_fam.any()) and (
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    r0.str.contains("promedio|plazo", na=False).any() or r1.str.contains("empres|famil|dias|días", na=False).any()
):
    df0 = df0.iloc[2:].reset_index(drop=True)

# ---- construir limpio ----
df_UI = df0[[fecha_col, emp_col, fam_col]].copy()
df_UI.columns = ["Fecha", "empresas", "familias"]


df_UI = (
    # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
    df_UI.dropna(subset=["Fecha"])
            # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
            .dropna(how="all", subset=["empresas","familias"])
            .sort_values("Fecha")
            .reset_index(drop=True)
)

# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_UI["Fecha"] = pd.to_datetime(df_UI["Fecha"], errors="coerce")+ MonthEnd(0)

ren = {
    "empresas":   "tasa_UI_empresas",
    "familias":    "tasa_UI_familias"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_UI = df_UI.rename(columns={k:v for k,v in ren.items() if k in df_UI.columns})
df_UI.columns.name = None

df_UI.head()


##Tasas en dólares
PATH  = "Bases originales/tasas_sistema_bancario.xls"
SHEET = "Activas U$S"

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def norm(s):
    if s is None or (isinstance(s,float) and pd.isna(s)): return ""
    s = str(s).strip().lower().replace("\n"," ")
    s = s.translate(str.maketrans("áéíóúÁÉÍÓÚñÑ","aeiouAEIOUnN"))
    return re.sub(r"\s+"," ",s)

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def excel_serial_to_ts(x):
    if pd.isna(x): return pd.NaT
    if isinstance(x,(int,float)) and 10000 <= x <= 60000:
        return pd.Timestamp("1899-12-30") + pd.to_timedelta(int(x), unit="D")
    return pd.NaT

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def to_float(s: pd.Series) -> pd.Series:
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    s = s.astype(str).str.replace("\u00A0","", regex=False)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s = s.str.replace(r"[^\d,.\-]", "", regex=True)
    # Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
    def fix(x):
        if "," in x and "." in x:  # '.' miles, ',' decimal
            return x.replace(".","").replace(",",".")
        if "," in x:
            if x.count(",")>1:
                L,R = x.rsplit(",",1);  return L.replace(",","")+"."+R
            return x.replace(",",".")
        return x
    return pd.to_numeric(s.apply(fix), errors="coerce")

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def parse_fecha(col: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(col): 
        return col
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    dt = pd.to_datetime(col, errors="coerce", dayfirst=True)
    miss = dt.isna()
    if miss.any():
        dt.loc[miss] = col[miss].apply(excel_serial_to_ts)
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    return pd.to_datetime(dt, errors="coerce")

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def flatten_two_level(mi) -> list[str]:
    out=[]
    for a,b in mi.to_list():
        # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        aa,bb = norm(a), norm(b)
        parts=[p for p in (aa,bb) if p and not p.startswith("unnamed")]
        out.append("__".join(parts) if parts else (aa or bb))
    return out

# ---- cargar crudo y detectar fila 'Mes' + encabezado doble ----
xls = pd.ExcelFile(PATH)
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel(xls, sheet_name=SHEET, header=None, dtype=object)

mes_rows = []
for i in range(min(len(raw), 160)):
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    row_norm = [norm(v) for v in raw.iloc[i].tolist()]
    if any(c == "mes" or c.startswith("fecha") for c in row_norm):
        mes_rows.append(i)

if not mes_rows:
    raise ValueError("No encontré ninguna fila con 'Mes' o 'Fecha'.")

chosen = None
two_level = False
for r in mes_rows:
    if r-1 >= 0:
        # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        prev = [norm(v) for v in raw.iloc[r-1].tolist()]
        if any("empres" in c for c in prev) or any("famil" in c for c in prev):
            chosen = (r-1, r); two_level=True; break
if chosen is None:
    chosen = (mes_rows[0],)

# ---- leer y aplanar ----
if two_level:
    # Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
    df0 = pd.read_excel(xls, sheet_name=SHEET, header=list(chosen))
    # Reutilización: aplicamos `flatten_two_level()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    flat = flatten_two_level(df0.columns)
    df0.columns = flat
else:
    # Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
    df0 = pd.read_excel(xls, sheet_name=SHEET, header=chosen[0])

# ---- elegir columnas clave (con heurística por contenido) ----
# Fecha = 'mes' o 'fecha'
# Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
fecha_candidates = [c for c in df0.columns if norm(c) == "mes" or norm(c).startswith("fecha")]
if not fecha_candidates:
    raise RuntimeError("No hallé columna de fecha ('Mes' o 'Fecha').")
fecha_col = fecha_candidates[0]

# Heurística: las filas 0–1 contienen rótulos ("promedio"/"empresas"/"familias"/"plazo ...")
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
r0 = df0.iloc[0].astype(str).str.lower()
# Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
r1 = df0.iloc[1].astype(str).str.lower()

# Columna de PROMEDIO EMPRESAS: fila0 contiene "promedio" y fila1 contiene "empres"
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
mask_emp = r0.str.contains("promedio", na=False) & r1.str.contains("empres", na=False)
emp_guess = list(df0.columns[mask_emp])

# Columna de PROMEDIO FAMILIAS: fila0 contiene "promedio" y fila1 contiene "famil"
# Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
mask_fam = r0.str.contains("promedio", na=False) & r1.str.contains("famil", na=False)
fam_guess = list(df0.columns[mask_fam])

# Si la heurística no detectó alguna, usa tu pick_group (con ranking)
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def pick_group(df, group, prefer_general=False):
    g = group.lower()
    names = list(df.columns)
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    n = lambda c: norm(c)

    cols = [c for c in names if n(c).startswith(g)]
    if not cols:
        return None

    exact = [c for c in cols if n(c) == g]
    if exact:
        return exact[0]

    prom = [c for c in cols if "prom" in n(c)]
    if prom:
        return sorted(prom, key=lambda c: len(n(c)))[0]

    bad_words = ("plazo","credito","crédito","creditos","créditos","sobregiro","sobregiros",
                "prestamo","prestamos","por")
    good = [c for c in cols if not any(w in n(c) for w in bad_words)]
    if good:
        return sorted(good, key=lambda c: len(n(c)))[0]

    if prefer_general:
        gen = next((c for c in names if n(c) == "promedio"), None)
        if gen:
            return gen
    return None

# Elegimos columnas finales
# Reutilización: aplicamos `pick_group()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
emp_col = (emp_guess[0] if emp_guess else pick_group(df0, "empresas", prefer_general=False))
# Reutilización: aplicamos `pick_group()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
fam_col = (fam_guess[0] if fam_guess else pick_group(df0, "familias", prefer_general=True))

if emp_col is None or fam_col is None:
    print("[diag] columnas aplanadas:")
    for c in df0.columns: print(" -", c)
    raise RuntimeError("No pude identificar la columna 'promedio' de empresas y/o familias.")

print(f"[sel] fecha_col='{fecha_col}', emp_col='{emp_col}', fam_col='{fam_col}'")

# ---- borrar las dos filas de rótulos si efectivamente aparecieron ----
if (mask_emp.any() or mask_fam.any()) and (
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    r0.str.contains("promedio|plazo", na=False).any() or r1.str.contains("empres|famil|dias|días", na=False).any()
):
    df0 = df0.iloc[2:].reset_index(drop=True)

# ---- construir limpio ----
df_dolares = df0[[fecha_col, emp_col, fam_col]].copy()
df_dolares.columns = ["Fecha", "empresas", "familias"]


df_dolares = (
    # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
    df_dolares.dropna(subset=["Fecha"])
            # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
            .dropna(how="all", subset=["empresas","familias"])
            .sort_values("Fecha")
            .reset_index(drop=True)
)

# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_dolares["Fecha"] = pd.to_datetime(df_dolares["Fecha"], errors="coerce")+ MonthEnd(0)

ren = {
    "empresas":   "tasa_dolares_empresas",
    "familias":    "tasa_dolares_familias"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_dolares = df_dolares.rename(columns={k:v for k,v in ren.items() if k in df_dolares.columns})
df_dolares.columns.name = None

df_dolares.head()

##Armamos un dataframe final con todas las tasas

DB_TASAS = (
    df_pesos
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_UI, on="Fecha", how="left")
    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    .merge(df_dolares, on="Fecha", how="inner")
)

DB_TASAS=DB_TASAS[DB_TASAS["Fecha"] >= pd.Timestamp("2015-01-31")].copy()

DB_TASAS.to_excel("Datasets finales/TASAS_DB.xlsx", index=False, float_format="%.2f")



[sel] fecha_col='mes', emp_col='empresas', fam_col='familias__sobregiros (5).1'
[sel] fecha_col='mes', emp_col='empresas', fam_col='familias__sobregiros (5).1'
[sel] fecha_col='mes', emp_col='empresas', fam_col='familias__sobregiros (5).1'


## Circulante de valores

Obtenemos el valor total de los diferentes títulos instrumentos emitidos por el BCU

**Archivo(s)/fuente(s):**
- `Bases originales/Circulante_valores_oferta_publica.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [27]:
##Circulante de valores

# -------- Config --------
PATH  = "Bases originales/Circulante_valores_oferta_publica.xlsx"
SHEET = "CIRCULANTE"  # ajusta si el nombre difiere

# -------- Helpers --------
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def norm(s):
    if s is None or (isinstance(s,float) and pd.isna(s)): return ""
    s = str(s).strip().lower().replace("\n"," ")
    s = s.translate(str.maketrans("áéíóúÁÉÍÓÚñÑ","aeiouAEIOUnN"))
    return re.sub(r"\s+"," ",s)

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def excel_serial_to_ts(x):
    if pd.isna(x): return pd.NaT
    if isinstance(x,(int,float)) and 10000 <= x <= 60000:
        return pd.Timestamp("1899-12-30") + pd.to_timedelta(int(x), unit="D")
    return pd.NaT

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def parse_fecha_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(s):
        dt = s
    else:
        # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
        dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
        miss = dt.isna()
        if miss.any():
            dt.loc[miss] = s[miss].apply(excel_serial_to_ts)
    return dt

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def to_float_series(s: pd.Series) -> pd.Series:
    if s.dtype.kind in "if": 
        # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
        return s.astype(float)
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    s = s.astype(str).str.strip()
    s = s.replace({"-": np.nan, "": np.nan})
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s = s.str.replace(r"[^\d\.,\-]", "", regex=True)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    both = s.str.contains(r"\.", na=False) & s.str.contains(r",", na=False)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s.loc[both] = s.loc[both].str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    only_comma = s.str.contains(",", na=False) & ~s.str.contains(r"\.", na=False)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s.loc[only_comma] = s.loc[only_comma].str.replace(",", ".", regex=False)
    return pd.to_numeric(s, errors="coerce")

# -------- 1) Localizar la fila del BLOQUE y leer con dos niveles --------
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel(PATH, sheet_name=SHEET, header=None, dtype=object)

hdr_top = None
for i in range(0, min(len(raw), 300)):
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    row = [norm(v) for v in raw.iloc[i].tolist()]
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    if any(norm(c) == "emision sector publico" for c in row):
        hdr_top = i
        break

if hdr_top is None:
    raise RuntimeError("No pude localizar la fila con 'Emisión Sector Público'.")

hdr_sub = hdr_top + 1  # la fila siguiente trae los subtítulos
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df0 = pd.read_excel(PATH, sheet_name=SHEET, header=[hdr_top, hdr_sub])

# Rellenar celdas combinadas en el header
mi = df0.columns
fixed = []
last = ["",""]
for a,b in mi.to_list():
    aa = a if (pd.notna(a) and not str(a).lower().startswith("unnamed")) else last[0]
    bb = b if (pd.notna(b) and not str(b).lower().startswith("unnamed")) else last[1]
    fixed.append((aa,bb))
    last = [aa,bb]
df0.columns = pd.MultiIndex.from_tuples(fixed)

# -------- 2) Detectar columna Fecha por contenido (no por texto) --------
fecha_col = None
best_rate = -1
for col in df0.columns:
    # ignorar columnas claramente numéricas del bloque
    s = df0[col]
    # Reutilización: aplicamos `parse_fecha_series()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    dt = parse_fecha_series(s)
    rate = dt.notna().mean()
    if rate >= 0.7 and rate > best_rate:  # “columna con fechas” razonable
        fecha_col = col
        best_rate = rate

if fecha_col is None:
    # fallback: primera columna
    fecha_col = df0.columns[0]

# -------- 3) Seleccionar TODAS las columnas del bloque 'Emisión Sector Público' --------
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def is_pub(col):
    a,b = col
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    return "emision sector publico" in norm(a)

# Reutilización: aplicamos `is_pub()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
pub_cols = [c for c in df0.columns if is_pub(c)]
if not pub_cols:
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    topvals = pd.Series([norm(c[0]) for c in df0.columns]).value_counts().head(20)
    raise RuntimeError(f"No encontré columnas bajo 'Emisión Sector Público'. Top nivel superior:\n{topvals}")

df_circulantes = df0[[fecha_col] + pub_cols].copy()

# Aplanar nombres
new_names = []
seen = {}
for a,b in df_circulantes.columns:
    if (a,b) == fecha_col:
        new_names.append("Fecha")
        continue
    # Reutilización: aplicamos `norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    base = norm(b) or norm(a) or "col"
    base = (base.replace("/", "_")
            .replace("%", "pct")
            .replace("(", "").replace(")", ""))
    if base in seen:
        seen[base] += 1
        new_names.append(f"{base}__{seen[base]}")
    else:
        seen[base] = 1
        new_names.append(base)

df_circulantes.columns = new_names

# -------- 4) Tipos y limpieza --------
# Reutilización: aplicamos `parse_fecha_series()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
df_circulantes["Fecha"] = parse_fecha_series(df_circulantes["Fecha"]).dt.to_period("M").dt.to_timestamp("M")
for c in df_circulantes.columns:
    if c == "Fecha": 
        continue
    # Reutilización: aplicamos `to_float_series()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    df_circulantes[c] = to_float_series(df_circulantes[c])

num_cols = [c for c in df_circulantes.columns if c != "Fecha"]
df_circulantes = (df_circulantes
            # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
            .dropna(subset=["Fecha"])
            # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
            .dropna(how="all", subset=num_cols)
            .sort_values("Fecha")
            .reset_index(drop=True))


keep = ["Fecha",
        "bonos del tesoro _ previsional",
        "letras de tesoreria _ regulacion monetaria",
        "notas bcu _ notas del tesoro",
        "obligaciones negociables",
        "fideicomisos financieros deuda",
        "emisiones internacionales"
        ]
df_circulantes=df_circulantes[keep].copy()

ren = {
    "bonos del tesoro _ previsional":   "valor_tot_bonos_del_tesoro",
    "letras de tesoreria _ regulacion monetaria":    "valor_tot_letras_de_tesoreria",
    "notas bcu _ notas del tesoro": "valor_tot_notas_del_tesoro",
    "fideicomisos financieros deuda" : "valor_total_fideicomisos_financieros_deuda",
    "emisiones internacionales": "valor_tot_emisiones_internacionales",
    "obligaciones negociables": "valor_tot_obl_negociables"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_circulantes = df_circulantes.rename(columns={k:v for k,v in ren.items() if k in df_circulantes.columns})
df_circulantes.columns.name = None
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_circulantes["Fecha"] = pd.to_datetime(df_circulantes["Fecha"], errors="coerce")+ MonthEnd(0)
df_circulantes.head()

DB_CIRCULANTES=df_circulantes

DB_CIRCULANTES=DB_CIRCULANTES[DB_CIRCULANTES["Fecha"] >= pd.Timestamp("2010-01-31")].copy()

DB_CIRCULANTES.to_excel("Datasets finales/CIRCULANTE_DB.xlsx", index=False, float_format="%.2f")



## Tasa call a un día

Proxy de condiciones monetarias de corto plazo.

**Archivo(s)/fuente(s):**
- `Bases originales/TasaCallHistoricoDiario.xlsx`

**Columnas referenciadas/seleccionadas en transformaciones:**
FECHA, PROMEDIO

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [28]:
##Tasa call a un dia (proxy de la TPM)

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_call=pd.read_excel("Bases originales/TasaCallHistoricoDiario.xlsx", sheet_name=0, header=0)
df_call=df_call[["FECHA","PROMEDIO"]].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_call=df_call.rename(columns={"PROMEDIO": "tasa_call_1dia"})
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_call=df_call.rename(columns={"FECHA": "Fecha"})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_call["Fecha"] = pd.to_datetime(df_call["Fecha"], errors="coerce")
df_call.head()

DB_CALL=df_call
DB_CALL.to_excel("Datasets finales/CALL_DB.xlsx", index=False, float_format="%.2f")


## Compras netas de divisas

Intervenciones/operaciones netas por sector.

**Archivo(s)/fuente(s):**
- `Bases originales/compras_netas_divisas_sectores.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [29]:
##Compras netas de divisas

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_divisas=pd.read_excel("Bases originales/compras_netas_divisas_sectores.xlsx", sheet_name=0, header=9)
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_divisas["Fecha"] = pd.to_datetime(df_divisas["Fecha"], errors="coerce")+ MonthEnd(0)
df_divisas=df_divisas.drop(columns=["RM"])
ren = {
    "EEPP":   "compras_div_empresas_publicas",
    "BCU":    "compras_div_BCU",
    "GG": "compras_div_gob_central",
    "FAP" : "compras_div_AFAP",
    "Familias-Empresas": "compras_div_fam_y_emp",
    "Bancos": "compras_div_bancos"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_divisas = df_divisas.rename(columns={k:v for k,v in ren.items() if k in df_divisas.columns})
df_divisas.head()
DB_DIVISAS=df_divisas
DB_DIVISAS.to_excel("Datasets finales/DIVISAS_DB.xlsx", index=False, float_format="%.2f")


## Venta de autos

Indicador de demanda/consumo durable.

**Archivo(s)/fuente(s):**
- `Bases originales/venta_autos.xls`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [30]:
##Venta de autos

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_autos=pd.read_excel("Bases originales/venta_autos.xls", sheet_name=0, header=0)
ren = {
    "FECHA":   "Fecha",
    "Total":    "venta_autos"   
    }
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_autos = df_autos.rename(columns={k:v for k,v in ren.items() if k in df_autos.columns})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_autos["Fecha"] = pd.to_datetime(df_autos["Fecha"], errors="coerce")+ MonthEnd(0)
df_autos.head()
DB_AUTOS=df_autos
DB_AUTOS.to_excel("Datasets finales/AUTOS_DB.xlsx", index=False, float_format="%.2f")


## Emisiones de letras (BEVSA)

Colocaciones/emisiones en mercado local.

**Archivo(s)/fuente(s):**
- `Bases originales/emisiones_letras_bevsa.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [31]:
##Emisiones de letras

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_letras=pd.read_excel("Bases originales/emisiones_letras_bevsa.xlsx", sheet_name=0, header=0)

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_letras=df_letras.rename(columns={"fecha_licitacion": "Fecha"})

# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_letras["Fecha"] = pd.to_datetime(df_letras["Fecha"], errors="coerce")

df_letras.head()

DB_LETRAS=df_letras
DB_LETRAS.to_excel("Datasets finales/LETRAS_DB.xlsx", index=False, float_format="%.2f")


## Licitaciones de letras/notas

Calendario y montos licitados.

**Archivo(s)/fuente(s):**
- `Bases originales/licitaciones_letras_notas.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [32]:
##Licitaciones de notas

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_licitaciones=pd.read_excel("Bases originales/licitaciones_letras_notas.xlsx", sheet_name=0, header=0)

ren = {
    "FECHA":   "Fecha",
    "VENCIMIENTO":    "total_vencimientos_titulos_gob",
    "ADJUDICADO": "total_adjudicado_titulos_gob",
    "LICITADO": "total_licitado_titulos_gob"
    }
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_licitaciones = df_licitaciones.rename(columns={k:v for k,v in ren.items() if k in df_licitaciones.columns})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_licitaciones["Fecha"] = pd.to_datetime(df_licitaciones["Fecha"], errors="coerce")
df_licitaciones.head()

DB_LICITACIONES=df_licitaciones
DB_LICITACIONES.to_excel("Datasets finales/LICITACIONES_DB.xlsx", index=False, float_format="%.2f")


## Operativa en mercados (BEVSA)

Volumen operado por mercado/instrumento.

**Archivo(s)/fuente(s):**
- `Bases originales/operativa_mercados_bevsa.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [33]:
##Operativa total en los diferentes mercados

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_operativa=pd.read_excel("Bases originales/operativa_mercados_bevsa.xlsx", sheet_name=0, header=0)

ren = {
    "Cambios":    "operativa_total_mercado_de_cambios",
    "Valores": "operativa_total_mercado_de_valores",
    "Dinero": "operativa_total_mercado_de_dinero"
    }
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_operativa = df_operativa.rename(columns={k:v for k,v in ren.items() if k in df_operativa.columns})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_operativa["Fecha"] = pd.to_datetime(df_operativa["Fecha"], errors="coerce") + MonthEnd(0)
df_operativa.head()

DB_OPERATIVA=df_operativa
DB_OPERATIVA.to_excel("Datasets finales/OPERATIVA_DB.xlsx", index=False, float_format="%.2f")


## Balanza de pagos

Cuenta corriente/financiera: variables del sector externo.

**Archivo(s)/fuente(s):**
- `Bases originales/balanza_pagos.xlsx`

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [34]:
# === Parámetros ===
path = "Bases originales/balanza_pagos.xlsx"   # o "/mnt/data/balanza_pagos.xlsx"
sheet_name = "Cuadro Nº 1"                      # cambie si desea otra hoja

# === Match de trimestres: admite '2012.I', '2025*.IV' y también 'IV 2025*' ===
pat_quarter = re.compile(
    r"^(?:\s*(?P<y1>\d{4})\*?\.(?P<q1>I{1,3}|IV)\s*|\s*(?P<q2>I{1,3}|IV)\s*-?\s*(?P<y2>\d{4})\*?\s*)$",
    re.IGNORECASE,
)

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def is_quarter_cell(x) -> bool:
    return isinstance(x, str) and bool(pat_quarter.match(x.strip()))

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def quarter_to_end(lbl: str) -> pd.Timestamp | None:
    if not isinstance(lbl, str):
        return None
    s = lbl.strip()
    m = pat_quarter.match(s)
    if not m:
        return None
    if m.group("y1"):
        y, qlab = int(m.group("y1")), m.group("q1")
    else:
        y, qlab = int(m.group("y2")), m.group("q2")
    q = {"I":1, "II":2, "III":3, "IV":4}[qlab.upper()]
    # Fin de trimestre
    return pd.Period(f"{y}Q{q}").to_timestamp("Q")

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def collapse_text_row(row) -> str:
    parts = []
    for v in row:
        if pd.notna(v):
            s = str(v).strip()
            if s:
                parts.append(s)
    return re.sub(r"\s+", " ", " ".join(parts)).strip()

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def clean_concept(s: str) -> str:
    s = str(s).strip()
    # Quitar códigos al inicio: P.3, B.1b, etc.
    s = re.sub(r"^\s*(?:[A-Za-z]\.\d+[A-Za-z]?|P\.\d+[A-Za-z]?)\s+", "", s)
    s = s.replace("*", "")       # quitar asteriscos
    s = s.replace("(-) ", "")    # opcional
    s = re.sub(r"\s+", " ", s).strip()
    return s

# === Leer hoja ===
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel(path, sheet_name=sheet_name, header=None, dtype=object)

# 1) Detectar la fila con más etiquetas de trimestre (en este archivo es la fila con '2012.I', '2012.II', ...)
quarter_counts_by_row = raw.applymap(is_quarter_cell).sum(axis=1)
r_qhead = int(quarter_counts_by_row.idxmax())

# 2) Ubicar el bloque continuo de columnas de fechas en esa fila
quarter_mask = raw.iloc[r_qhead].apply(is_quarter_cell)
quarter_cols_idx = np.where(quarter_mask.values)[0].tolist()
if not quarter_cols_idx:
    raise ValueError("No encontré etiquetas de trimestre.")
left_limit, right_limit = min(quarter_cols_idx), max(quarter_cols_idx) + 1

# 3) Parsear etiquetas → Timestamps (fin de trimestre)
date_labels = raw.iloc[r_qhead, left_limit:right_limit].tolist()
# Reutilización: aplicamos `quarter_to_end()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
date_cols = [quarter_to_end(x) for x in date_labels]

# 4) Valores debajo + “Concepto” a la izquierda
values_block = raw.iloc[r_qhead+1:, left_limit:right_limit].copy()
values_block.columns = date_cols
text_block = raw.iloc[r_qhead+1:, :left_limit].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
concept = text_block.apply(collapse_text_row, axis=1).rename("Concepto")

# 5) Armar, limpiar y pasar a largo
# Consolidación: apilamos fuentes con estructura compatible (mismo set de columnas) para una tabla única.
df_balanza = pd.concat([concept, values_block], axis=1)
fecha_cols = [c for c in df_balanza.columns if isinstance(c, pd.Timestamp)]
df_balanza = (
    df_balanza.loc[:, ["Concepto"] + fecha_cols]
            # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
            .dropna(axis=1, how="all")
            # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
            .dropna(axis=0, how="all")
            .reset_index(drop=True)
)

# Reestructuración: pasamos de formato ancho a largo para facilitar merges por fecha y tratamiento de variables.
balanza_tidy = df_balanza.melt(id_vars="Concepto", var_name="Fecha", value_name="Valor")
balanza_tidy["Valor"] = pd.to_numeric(balanza_tidy["Valor"], errors="coerce")
# Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
balanza_tidy = balanza_tidy.dropna(subset=["Fecha", "Valor"]).reset_index(drop=True)
balanza_tidy["Concepto"] = balanza_tidy["Concepto"].map(clean_concept)

# 6) Ancho (una columna por variable), index por Fecha
balanza_wide = (
    # Reestructuración: pivot para volver a formato ancho cuando conviene (una columna por variable).
    balanza_tidy.pivot_table(index="Fecha", columns="Concepto", values="Valor", aggfunc="first")
                .sort_index()
                .reset_index()
)
balanza_wide.columns.name = None

# Ejemplo de guardado
# balanza_wide.to_csv("balanza_wide_cuadro1.csv", index=False)

print("Hoja:", sheet_name)
print("Fila cabecera trimestres:", r_qhead)
print("Rango de fechas:", balanza_tidy["Fecha"].min(), "→", balanza_tidy["Fecha"].max())
print("Variables:", len(balanza_wide.columns) - 1, " |  Fechas:", balanza_wide.shape[0])
balanza_wide.head()

cols_keep = [
    "Fecha",
    "1. Cuenta Corriente",
    "1.B Ingreso Primario (2)",
    "3.1 Inversión directa",
    "3.2 Inversión de cartera",
    "3.3 Derivados financieros (distintos de reservas)",
    "Adquisición neta de activos financieros"
]

# Crear nuevo DataFrame solo con esas columnas
balanza_wide = balanza_wide[cols_keep].copy()

ren = {
    "1. Cuenta Corriente":   "saldo_cuenta_corriente",
    "1.B Ingreso Primario (2)":    "ingreso_primario",
    "3.1 Inversión directa": "inversion_directa",
    "3.2 Inversión de cartera" : "inversion_de_cartera",
    "Adquisición neta de activos financieros": "adq_neta_act_financieros",
    "3.3 Derivados financieros (distintos de reservas)": "derivados_financieros"
}

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
balanza_wide = balanza_wide.rename(columns={k:v for k,v in ren.items() if k in balanza_wide.columns})

balanza_wide.head()

DB_BALANZA=balanza_wide
DB_BALANZA.to_excel("Datasets finales/BALANZA_DB.xlsx", index=False, float_format="%.2f")


Hoja: Cuadro Nº 1
Fila cabecera trimestres: 9
Rango de fechas: 2012-03-31 00:00:00 → 2025-03-31 00:00:00
Variables: 52  |  Fechas: 53


C:\Users\jbrio\AppData\Local\Temp\ipykernel_13548\1452643514.py:56: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  quarter_counts_by_row = raw.applymap(is_quarter_cell).sum(axis=1)


## Deuda Pública

Nos quedamos únicamente con la deuda pública total.

**Archivo(s)/fuente(s):**
- `Bases originales/Deuda_Publica_SPG.xls`

**Columnas referenciadas/seleccionadas en transformaciones:**
Fecha, Total

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [ ]:
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
import pandas as pd, numpy as np, re, unicodedata
# Dependencias: librerías para lectura, limpieza, estandarización y manipulación de series.
from pathlib import Path

# ================== Parámetros ==================
PATH  = "Bases originales/Deuda_Publica_SPG.xls"
SHEET = "SPG2"
HEADER_TOP_ROWS = 6

# ================== Utilidades ==================
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def _norm(s: object) -> str:
    if s is None or (isinstance(s, float) and pd.isna(s)): return ""
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r"[\u00A0\u2000-\u200B]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def looks_like_period(x: object) -> bool:
    if pd.isna(x): return False
    if isinstance(x, (int, float)) and 1800 <= int(x) <= 2100: return True
    # Reutilización: aplicamos `_norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    s = _norm(x)
    return any((
        bool(re.fullmatch(r"\d{4}\*?", s)),
        bool(re.fullmatch(r"(I{1,3}|IV)\s*\.?\s*\d{2}\*?", s, re.I)),
        bool(re.fullmatch(r"\d{4}\s*\.?\s*(I{1,3}|IV)\*?", s, re.I)),
        bool(re.fullmatch(r"(I{1,3}|IV)\s*\d{4}\*?", s, re.I)),
    ))

Q = {"I":1, "II":2, "III":3, "IV":4}

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def parse_periodo_cell(x: object) -> pd.Timestamp:
    if pd.isna(x): return pd.NaT
    if isinstance(x, (int, float)) and 1800 <= int(x) <= 2100:
        return pd.Period(f"{int(x)}Q4").to_timestamp("Q")
    # Reutilización: aplicamos `_norm()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    s = _norm(x)
    m = re.fullmatch(r"(\d{4})\*?", s)
    if m: return pd.Period(f"{int(m.group(1))}Q4").to_timestamp("Q")
    m = re.fullmatch(r"(I{1,3}|IV)\s*\.?\s*(\d{2})\*?", s, re.I)
    if m:
        q = Q[m.group(1).upper()]; yy = int(m.group(2))
        y = 2000 + yy if yy <= 50 else 1900 + yy
        return pd.Period(f"{y}Q{q}").to_timestamp("Q")
    m = re.fullmatch(r"(\d{4})\s*\.?\s*(I{1,3}|IV)\*?", s, re.I)
    if m:
        y = int(m.group(1)); q = Q[m.group(2).upper()]
        return pd.Period(f"{y}Q{q}").to_timestamp("Q")
    m = re.fullmatch(r"(I{1,3}|IV)\s*(\d{4})\*?", s, re.I)
    if m:
        q = Q[m.group(1).upper()]; y = int(m.group(2))
        return pd.Period(f"{y}Q{q}").to_timestamp("Q")
    return pd.NaT

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def build_colname(vals) -> str:
    toks = [t for t in map(_norm, vals) if t]
    if any(not re.fullmatch(r"\d+(\.\d+)?", t) for t in toks):
        toks = [t for t in toks if not re.fullmatch(r"\d+(\.\d+)?", t)]
    out = []
    for t in toks:
        if not out or out[-1] != t: out.append(t)
    return " — ".join(out)

# ================== Paso 1: detectar período y encabezado ==================
read0 = dict(sheet_name=SHEET, header=None)
if Path(PATH).suffix.lower() == ".xls":
    read0["engine"] = "xlrd"
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
raw = pd.read_excel(PATH, **read0)

best_col, best_cnt = None, -1
for j in range(raw.shape[1]):
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    cnt = raw.iloc[:, j].apply(looks_like_period).astype(int).sum()
    if cnt > best_cnt: best_col, best_cnt = j, cnt
if best_cnt <= 0:
    raise ValueError("No se encontró columna de períodos en SPG2.")

mask = raw.iloc[:, best_col].apply(looks_like_period).to_numpy()
start = int(np.flatnonzero(mask)[0])
top   = max(0, start - HEADER_TOP_ROWS)
hdr_block = raw.iloc[top:start, :]

# Reutilización: aplicamos `build_colname()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
col_names = [build_colname(hdr_block.iloc[:, j].tolist()) for j in range(raw.shape[1])]

# ================== Paso 2: leer con header real y renombrar ==================
head_row = start - 1
read1 = dict(sheet_name=SHEET, header=head_row)
if Path(PATH).suffix.lower() == ".xls":
    read1["engine"] = "xlrd"
# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df = pd.read_excel(PATH, **read1)

# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df = df.rename(columns={df.columns[best_col]: "Periodo"})
for j, c in enumerate(df.columns):
    if c != "Periodo":
        # Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
        df = df.rename(columns={c: (col_names[j] or str(c))})

# 👈 Duplicados: colapsar columnas repetidas tomando la PRIMERA no nula por fila
if df.columns.duplicated().any():
    # Agregación: resumimos a la granularidad necesaria (diaria/mensual/trimestral) para compatibilizar series.
    df = df.T.groupby(level=0).first().T

# ================== Paso 3: Fecha y limpieza ==================
df["Periodo_norm"] = df["Periodo"].map(_norm)
df["Fecha"] = df["Periodo_norm"].map(parse_periodo_cell)

value_cols = [c for c in df.columns if c not in ["Periodo", "Periodo_norm", "Fecha"]]

# convertir TODO el bloque de una: evita el problema Series/DataFrame
df[value_cols] = df[value_cols].apply(pd.to_numeric, errors="coerce")

df = df[df[value_cols].notna().any(axis=1)]
df = df[df["Fecha"].notna()].copy()

# ================== Paso 4: deduplicar por trimestre ==================
tmp = df.copy()
tmp["_nonnull"] = tmp[value_cols].notna().sum(axis=1)
# Agregación: resumimos a la granularidad necesaria (diaria/mensual/trimestral) para compatibilizar series.
idx = tmp.groupby("Fecha")["_nonnull"].idxmax()
df_quarters = tmp.loc[idx, ["Fecha"] + value_cols].sort_values("Fecha").reset_index(drop=True)

# (Opcional) completar todos los trimestres del rango:
# full_idx = pd.period_range(df_quarters["Fecha"].min(), df_quarters["Fecha"].max(), freq="Q").to_timestamp("Q")
# df_full = df_quarters.set_index("Fecha").reindex(full_idx).rename_axis("Fecha").reset_index()
# df_spg2_wide = df_full

df_spg2_wide = df_quarters

print("Filas (sin duplicados):", len(df_spg2_wide), "| columnas:", df_spg2_wide.shape[1])
print("Rango:", df_spg2_wide["Fecha"].min(), "→", df_spg2_wide["Fecha"].max())

df_spg2_wide.head()
df_spg2_wide=df_spg2_wide[["Fecha", "Total"]].copy()
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_spg2_wide=df_spg2_wide.rename(columns={"Total":"total_deuda_publica"})


DB_DEUDA=df_spg2_wide
DB_DEUDA.to_excel("Datasets finales/DEUDA_DB.xlsx", index=False, float_format="%.2f")



Filas (sin duplicados): 102 | columnas: 15
Rango: 1999-12-31 00:00:00 → 2025-03-31 00:00:00


## Dólar blue (Argentina)

Tipo de cambio paralelo ARS/USD (proxy de stress regional).

**Archivo(s)/fuente(s):**
- `Bases originales/dolar_blue.xlsx`

**Columnas referenciadas/seleccionadas en transformaciones:**
Fecha, Promedio

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [36]:
##Dolar Blue argentino

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_dolar_blue=pd.read_excel("Bases originales/dolar_blue.xlsx", sheet_name=0, header=0)

df_dolar_blue.head()
df_dolar_blue=df_dolar_blue[["Fecha", "Promedio"]].copy()
ren = {
    "Promedio": "dolar_blue_arg"
    }
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_dolar_blue = df_dolar_blue.rename(columns={k:v for k,v in ren.items() if k in df_dolar_blue.columns})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_dolar_blue["Fecha"] = pd.to_datetime(df_dolar_blue["Fecha"], errors="coerce")

DB_DOLARBLUE=df_dolar_blue
DB_DOLARBLUE.to_excel("Datasets finales/DOLARBLUE_DB.xlsx", index=False, float_format="%.2f")


## Histórico dólar (UYU/USD)

Tipo de cambio nominal en Uruguay.

**Archivo(s)/fuente(s):**
- `Bases originales/HIST_Dolar.xlsx`

**Columnas referenciadas/seleccionadas en transformaciones:**
CIERRE BEVSA FONDO, FECHA

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [ ]:
##Evolución histórica del dolar

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_dolar=pd.read_excel("Bases originales/HIST_Dolar.xlsx", sheet_name=0, header=0)
df_dolar=df_dolar[["FECHA","CIERRE BCU BILLETE"]]
df_dolar=df_dolar.iloc[1:]
ren = {
    "FECHA":   "Fecha",
    "CIERRE BEVSA FONDO":    "dolar"   
    }
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_dolar = df_dolar.rename(columns={k:v for k,v in ren.items() if k in df_dolar.columns})
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_dolar["Fecha"] = pd.to_datetime(df_dolar["Fecha"], errors="coerce")
df_dolar.head()
DB_DOLAR=df_dolar
DB_DOLAR=DB_DOLAR[DB_DOLAR["Fecha"] >= pd.Timestamp("2010-01-01")].copy()
DB_DOLAR.to_excel("Datasets finales/DOLAR_DB.xlsx", index=False, float_format="%.2f")


## Operaciones dólar (mercado local)

Volumen/operativa diaria en mercado de cambios.

**Archivo(s)/fuente(s):**
- `Bases originales/HistoricoDiario_Operaciones_Dolar.xlsx`

**Columnas referenciadas/seleccionadas en transformaciones:**
CANTIDAD, CANTIDAD_OPERACIONES, FECHA, OFERTA_COMPRA, OFERTA_VENTA

**Objetivo del bloque:** dejar la(s) serie(s) en un formato consistente (fecha como clave, nombres estandarizados, unidades homogéneas) para poder integrarlas en el dataset final.


In [38]:
##Operativa del mercado de dólares

# Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
df_operativa_dolar=pd.read_excel("Bases originales/HistoricoDiario_Operaciones_Dolar.xlsx", sheet_name=0, header=0)

# Renombrar columnas de compra/venta
# Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
df_operativa_dolar = df_operativa_dolar.rename(columns={
    "MEJORES OFERTAS FONDO": "OFERTA_COMPRA",
    "Unnamed: 5": "OFERTA_VENTA",
    "CANTIDAD OP.": "CANTIDAD_OPERACIONES",
    "CANTIDAD": "CANTIDAD"
})

# Asegurar que FECHA sea datetime
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_operativa_dolar["FECHA"] = pd.to_datetime(df_operativa_dolar["FECHA"], errors="coerce")

# Filtrar hasta el 5 de septiembre (de cualquier año)
# Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
df_filtrado = df_operativa_dolar[df_operativa_dolar["FECHA"] <= pd.to_datetime("2025-09-05")]

# Quedarse solo con las columnas necesarias
df_operativa_dolar = df_filtrado[[
    "FECHA",
    "OFERTA_COMPRA",
    "OFERTA_VENTA",
    "CANTIDAD_OPERACIONES",
    "CANTIDAD"
]]
df_operativa_dolar.tail(200)
DB_OPERATIVA_DOLAR=df_operativa_dolar
DB_OPERATIVA_DOLAR.to_excel("Datasets finales/OPERATIVA_DOLAR_DB.xlsx", index=False, float_format="%.2f")


## Armamos el dataset final

Hacemos un merge de todas las bases previamente armadas para obtener el dataset final, con el cual se llevará adelante el trabajo


In [39]:
##Armamos el dataset final

# ========= CONFIG =========
FOLDER = r"Datasets finales"       # <--- carpeta con tus archivos
SAVE_AS = r"TC_final.xlsx"     # salida opcional
ADD_FILE_PREFIX = False              # prefijar columnas con el nombre de archivo

# ========= HELPERS =========
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def norm_col(s: str) -> str:
    s = re.sub(r"\s+", " ", str(s).strip())
    s = s.translate(str.maketrans("áéíóúÁÉÍÓÚñÑ", "aeiouAEIOUnN"))
    return s

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def excel_serial_to_ts(x):
    if pd.isna(x): return pd.NaT
    if isinstance(x,(int,float)) and 10000 <= x <= 60000:
        return pd.Timestamp("1899-12-30") + pd.to_timedelta(int(x), unit="D")
    return pd.NaT

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def parse_any_date(s: pd.Series) -> pd.Series:
    """Parsea fechas de forma robusta (string o serial Excel)."""
    if pd.api.types.is_datetime64_any_dtype(s):
        dt = s
    else:
        # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
        dt = pd.to_datetime(s, errors="coerce", dayfirst=True)
        miss = dt.isna()
        if miss.any():
            dt.loc[miss] = s[miss].apply(excel_serial_to_ts)
    # quitar hora si viene (normalizar a medianoche)
    # Tiempo como clave: convertimos a datetime para alinear todas las series por fecha (merge temporal).
    return pd.to_datetime(dt.dt.date, errors="coerce")

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def infer_frequency(dt: pd.Series) -> str:
    """
    Heurística simple:
      - 'M' si ~>80% es fin de mes (y pocos días distintos)
      - 'Q' si ~>80% es fin de trimestre
      - 'D' si el delta mediano <= 3 días (serie “densa”)
      - 'U' desconocida en otro caso (no se toca)
    """
    # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
    d = dt.dropna().sort_values()
    if d.empty:
        return "U"

    # señales de mensual / trimestral
    is_me = d.dt.is_month_end
    is_qe = d.dt.is_quarter_end
    day_count = d.dt.day.nunique()

    if is_me.mean() >= 0.8 and day_count <= 5:
        return "M"
    if is_qe.mean() >= 0.8 and day_count <= 5:
        return "Q"

    # densidad diaria
    # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
    deltas = d.diff().dropna()
    if not deltas.empty and deltas.median() <= pd.Timedelta(days=3):
        return "D"

    return "U"  # desconocida

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def normalize_dates_by_freq(dt: pd.Series, freq: str) -> pd.Series:
    """Normaliza fechas según frecuencia detectada."""
    if freq == "M":
        return dt.dt.to_period("M").dt.to_timestamp("M")
    if freq == "Q":
        return dt.dt.to_period("Q").dt.to_timestamp("Q")
    # 'D' o 'U' -> no tocar (ya se quitó la hora en parse_any_date)
    return dt

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def to_float_series(s: pd.Series) -> pd.Series:
    """Convierte strings con separadores a float."""
    if s.dtype.kind in "if":
        # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
        return s.astype(float)
    # Tipos: forzamos dtypes para evitar merges fallidos y asegurar cálculos consistentes (p.ej., floats vs strings).
    s = s.astype(str).str.strip()
    s = s.replace({"-": np.nan, "": np.nan})
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s = s.str.replace(r"[^\d\.,\-]", "", regex=True)

    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    both = s.str.contains(r"\.", na=False) & s.str.contains(r",", na=False)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s.loc[both] = s.loc[both].str.replace(".", "", regex=False).str.replace(",", ".", regex=False)

    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    only_comma = s.str.contains(",", na=False) & ~s.str.contains(r"\.", na=False)
    # Limpieza de texto: normalizamos strings (espacios, mayúsculas/minúsculas) para evitar categorías duplicadas.
    s.loc[only_comma] = s.loc[only_comma].str.replace(",", ".", regex=False)

    return pd.to_numeric(s, errors="coerce")

# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def load_one_file(path: str) -> pd.DataFrame | None:
    """Lee un archivo y devuelve df con Fecha + series (n>=1), normalizando la fecha por frecuencia."""
    ext = Path(path).suffix.lower()
    name = Path(path).stem

    try:
        if ext in {".csv"}:
            # Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
            df0 = pd.read_csv(path)
        elif ext in {".xlsx", ".xls"}:
            # Ingesta: cargamos la fuente y la transformamos a un DataFrame para poder homogenizar nombres, tipos y frecuencia.
            df0 = pd.read_excel(path, sheet_name=0)
        elif ext in {".parquet"}:
            df0 = pd.read_parquet(path)
        else:
            print(f"  (skip) {path} → extensión no soportada")
            return None
    except Exception as e:
        print(f"  (error) {path}: {e}")
        return None

    # normalizar nombres
    # Reutilización: aplicamos `norm_col()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    df0.columns = [norm_col(c) for c in df0.columns]

    # ubicar columna Fecha
    fecha_candidates = [c for c in df0.columns if c.lower() == "fecha"]
    if not fecha_candidates:
        print(f"  (skip) {path} → no encontré columna 'Fecha'")
        return None

    fecha_col = fecha_candidates[0]
    df0 = df0.copy()
    # Reutilización: aplicamos `parse_any_date()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    dt = parse_any_date(df0[fecha_col])

    # inferir frecuencia y normalizar solo si corresponde
    # Reutilización: aplicamos `infer_frequency()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    freq = infer_frequency(dt)
    # Reutilización: aplicamos `normalize_dates_by_freq()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
    dt_norm = normalize_dates_by_freq(dt, freq)

    # columnas de datos
    data_cols = [c for c in df0.columns if c not in (fecha_col, "Fecha")]
    if not data_cols:
        print(f"  (skip) {path} → no hay columnas de datos además de 'Fecha'")
        return None

    df = pd.DataFrame({"Fecha": dt_norm})
    for c in data_cols:
        # Reutilización: aplicamos `to_float_series()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        df[c] = to_float_series(df0[c])

    # limpiar
    # Calidad de datos: tratamos nulos según criterio (evitar sesgos o rupturas en el armado final).
    df = df.dropna(subset=["Fecha"]).dropna(how="all", subset=data_cols)
    if df.empty:
        print(f"  (skip) {path} → no quedó contenido tras limpieza")
        return None

    # prefijo para evitar choques
    if ADD_FILE_PREFIX:
        # Estandarización: unificamos nombres de columnas (clave para merges reproducibles y legibles).
        df = df.rename(columns={c: f"{name}_{c}" for c in data_cols})

    # agrupar por Fecha por si viniera duplicada
    agg = {c: "first" for c in df.columns if c != "Fecha"}
    # Agregación: resumimos a la granularidad necesaria (diaria/mensual/trimestral) para compatibilizar series.
    df = df.groupby("Fecha", as_index=False).agg(agg).sort_values("Fecha")

    print(f"  (ok) {path}: freq={freq} | cols={', '.join([c for c in df.columns if c!='Fecha'])}")
    return df

# ========= MAIN =========
# Función auxiliar: encapsula una regla de limpieza para reutilizarla y asegurar consistencia entre datasets.
def merge_all(folder: str) -> pd.DataFrame:
    pattern = os.path.join(folder, "*")
    files = [p for p in glob.glob(pattern) if Path(p).suffix.lower() in {".csv",".xls",".xlsx",".parquet"}]
    if not files:
        raise RuntimeError("No se encontraron archivos en la carpeta indicada.")

    dfs = []
    print(f"Archivos encontrados: {len(files)}")
    for f in sorted(files):
        # Reutilización: aplicamos `load_one_file()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
        df = load_one_file(f)
        if df is not None:
            dfs.append(df)

    if not dfs:
        raise RuntimeError("No se pudo cargar ninguna serie (revisa mensajes).")

    # Integración: combinamos fuentes por claves (usualmente fecha); el tipo de join determina el universo temporal final.
    merged = reduce(lambda l, r: pd.merge(l, r, on="Fecha", how="outer"), dfs)
    merged = merged.sort_values("Fecha").reset_index(drop=True)
    return merged

# Ejecutar
# Reutilización: aplicamos `merge_all()` para mantener el *mismo criterio* de limpieza/estandarización entre fuentes.
TC_final = merge_all(FOLDER)
TC_final=TC_final[TC_final["Fecha"] >= pd.Timestamp("2015-01-01")].copy()
print("\nPreview:")
print(TC_final.head())
print("\nShape:", TC_final.shape)

# Guardar (opcional)
try:
    TC_final.to_excel(SAVE_AS, index=False)
    print(f"\nGuardado en: {SAVE_AS}")
except Exception as e:
    print(f"(no se guardó) {e}")


Archivos encontrados: 35
  (ok) Datasets finales\ANCAP_DB.xlsx: freq=M | cols=compras_ANCAP, int_ANCAP, var_stock_pet, inv_ANCAP
  (ok) Datasets finales\AUTOS_DB.xlsx: freq=M | cols=venta_autos
  (ok) Datasets finales\BALANZA_DB.xlsx: freq=M | cols=saldo_cuenta_corriente, ingreso_primario, inversion_directa, inversion_de_cartera, derivados_financieros, adq_neta_act_financieros
  (ok) Datasets finales\BM_DB.xlsx: freq=M | cols=act_ext_sist_brio, cred_int_publ_sist_brio, cred_neto_al_BCU, cred_priv_res_sist_brio, LRM_sist_brio, act_ext_BCU, cred_int_publ_BCU, cred_neto_al_sist_brio, cred_priv_res_BCU, LRM_res_BCU, dep_plazo_MN, dep_ME
  (ok) Datasets finales\BRASIL_DB.xlsx: freq=D | cols=TPM_br, usd_br, IPC_br, IA_br, inflacion_12m_br, EPU_br
  (ok) Datasets finales\CALL_DB.xlsx: freq=D | cols=tasa_call_1dia
  (ok) Datasets finales\CIRCULANTE_DB.xlsx: freq=M | cols=valor_tot_bonos_del_tesoro, valor_tot_letras_de_tesoreria, valor_tot_notas_del_tesoro, valor_tot_obl_negociables, valor_tota